In [1]:
import pandas as pd

df = pd.read_csv('bootstrap_samples_5000x30_1.csv')
percentile_df = pd.read_csv('percentile_1.csv')
df_params = pd.read_csv('data_fullX30_1.csv')

In [2]:
import re
import numpy as np
import pandas as pd

# percentile.csv 전처리
score_table = percentile_df.copy()

AGE_SLEEP_RANGES = {
    '0-9': (9.0, 11.0),
    '10대': (8.0, 10.0),
    '20대': (7.0, 9.0),
    '30대': (7.0, 9.0),
    '40대': (7.0, 9.0),
    '50대': (7.0, 9.0),
    '60대': (7.0, 9.0),
    '70대': (7.0, 8.0),
    '80대+': (7.0, 8.0),
}
DEFAULT_SLEEP_LOW = 7.0
DEFAULT_SLEEP_HIGH = 9.0
EFFORT_DEFAULT_DAY = 30

EFFORT_ALPHA = 0.5
EFFORT_BETA = 0.2
EFFORT_GAMMA = 0.3
EFFORT_RESCALE = 1.5
EPS_DENOM = 1e-6


def normalize_age_group(raw):
    if pd.isna(raw):
        return np.nan
    text = str(raw).strip().replace(' ', '')
    if text == '0-9':
        return '0-9'
    digits = re.findall(r'\d+', text)
    if not digits:
        return text
    decade = int(digits[0])
    if '+' in text or decade >= 80:
        return '80대+'
    return f'{decade}대'


def age_to_group(age):
    if pd.isna(age):
        return np.nan
    value = float(age)
    if value < 10:
        return '0-9'
    if value >= 80:
        return '80대+'
    return f'{int(value // 10) * 10}대'


# 하위 셀 호환용(스칼라 함수)
def sleep_deviation_from_daily_sleep(sleep_hours, age_group):
    if pd.isna(sleep_hours):
        return np.nan
    low, high = AGE_SLEEP_RANGES.get(age_group, (DEFAULT_SLEEP_LOW, DEFAULT_SLEEP_HIGH))
    value = float(sleep_hours)
    if value < low:
        return low - value
    if value > high:
        return value - high
    return 0.0


def _vectorized_sleep_deviation(sleep_series, age_group_series):
    low_map = {k: v[0] for k, v in AGE_SLEEP_RANGES.items()}
    high_map = {k: v[1] for k, v in AGE_SLEEP_RANGES.items()}
    sleep_values = pd.to_numeric(sleep_series, errors='coerce').to_numpy(dtype=float)
    low_values = age_group_series.map(low_map).fillna(DEFAULT_SLEEP_LOW).to_numpy(dtype=float)
    high_values = age_group_series.map(high_map).fillna(DEFAULT_SLEEP_HIGH).to_numpy(dtype=float)
    return np.where(
        sleep_values < low_values,
        low_values - sleep_values,
        np.where(sleep_values > high_values, sleep_values - high_values, 0.0),
    )


def _safe_change_ratio(current_value, baseline_value):
    if pd.isna(current_value) or pd.isna(baseline_value):
        return np.nan
    base_value = float(baseline_value)
    if abs(base_value) < EPS_DENOM:
        return np.nan
    return (float(current_value) - base_value) / base_value


def _improvement_score_from_ratio(ratio):
    if pd.isna(ratio):
        return 50.0
    if ratio <= -0.10:
        return 20.0
    if ratio < 0.10:
        return 50.0
    if ratio < 0.20:
        return 70.0
    if ratio < 0.35:
        return 85.0
    return 100.0


def _mean_ignore_nan(values):
    finite_mask = np.isfinite(values)
    if not np.any(finite_mask):
        return np.nan
    return float(values[finite_mask].mean())


def _safe_rate_ge(values, threshold):
    if values.size == 0:
        return np.nan
    return float((values >= threshold).mean())


def _safe_rate_le(values, threshold):
    if values.size == 0:
        return np.nan
    return float((values <= threshold).mean())


score_table['age_group_norm'] = score_table['age_group'].apply(normalize_age_group)

required_cols = ['RIDAGEYR', 'DAILY_SLEEP', 'SLQ300', 'SLQ310', 'SLQ320', 'SLQ330', 'avg_daily_steps']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f'df에 필요한 컬럼이 없습니다: {missing_cols}')

scored_df = df.copy()
scored_df['age_group_for_score'] = scored_df['RIDAGEYR'].apply(age_to_group)
scored_df['sleep_deviation_value'] = _vectorized_sleep_deviation(scored_df['DAILY_SLEEP'], scored_df['age_group_for_score'])
scored_df['sleep_pattern_value'] = (
    scored_df['SLQ300']
    + scored_df['SLQ310'].abs()
    + scored_df['SLQ320']
    + scored_df['SLQ330']
)
scored_df['step_value'] = scored_df['avg_daily_steps']

score_table_valid = score_table[score_table['percentile_value'].notna()].copy()


def _build_percentile_lookup_fast(score_table_valid_df):
    lookup = {}
    grouped = score_table_valid_df.groupby(['domain', 'age_group_norm'], sort=False)
    for (domain, age_group), sub_df in grouped:
        ref_values_raw = pd.to_numeric(sub_df['percentile_value'], errors='coerce').to_numpy(dtype=float)
        ref_percentiles_raw = pd.to_numeric(sub_df['percentile'], errors='coerce').to_numpy(dtype=float)

        first_by_value = {}
        for idx, (ref_value, ref_pct) in enumerate(zip(ref_values_raw, ref_percentiles_raw)):
            if not np.isfinite(ref_value):
                continue
            if ref_value not in first_by_value:
                first_by_value[ref_value] = (idx, float(ref_pct))

        if not first_by_value:
            continue

        unique_values = np.array(sorted(first_by_value.keys()), dtype=float)
        unique_first_pos = np.array([first_by_value[v][0] for v in unique_values], dtype=np.int64)
        unique_percentiles = np.array([first_by_value[v][1] for v in unique_values], dtype=float)

        lookup[(domain, age_group)] = (unique_values, unique_percentiles, unique_first_pos)
    return lookup


percentile_lookup = _build_percentile_lookup_fast(score_table_valid)


def nearest_percentile_batch(domain, age_group_series, value_series):
    n_rows = len(value_series)
    out_percentile = np.full(n_rows, np.nan, dtype=float)
    out_ref_value = np.full(n_rows, np.nan, dtype=float)

    age_values = age_group_series.to_numpy()
    target_values = pd.to_numeric(value_series, errors='coerce').to_numpy(dtype=float)

    for age_group in pd.unique(age_group_series.dropna()):
        key = (domain, age_group)
        if key not in percentile_lookup:
            continue

        ref_values, ref_percentiles, ref_first_pos = percentile_lookup[key]
        if ref_values.size == 0:
            continue

        mask = (age_values == age_group) & np.isfinite(target_values)
        if not np.any(mask):
            continue

        target_pos = np.flatnonzero(mask)
        targets = target_values[target_pos]

        if ref_values.size == 1:
            out_percentile[target_pos] = ref_percentiles[0]
            out_ref_value[target_pos] = ref_values[0]
            continue

        insert_pos = np.searchsorted(ref_values, targets, side='left')
        right_idx = np.clip(insert_pos, 0, ref_values.size - 1)
        left_idx = np.clip(insert_pos - 1, 0, ref_values.size - 1)

        diff_left = np.abs(targets - ref_values[left_idx])
        diff_right = np.abs(targets - ref_values[right_idx])

        choose_right = (diff_right < diff_left) | (
            (diff_right == diff_left) & (ref_first_pos[right_idx] < ref_first_pos[left_idx])
        )
        chosen_idx = np.where(choose_right, right_idx, left_idx)

        out_percentile[target_pos] = ref_percentiles[chosen_idx]
        out_ref_value[target_pos] = ref_values[chosen_idx]

    return pd.Series(out_percentile, index=value_series.index), pd.Series(out_ref_value, index=value_series.index)


sleep_p, sleep_ref = nearest_percentile_batch('sleep_time', scored_df['age_group_for_score'], scored_df['sleep_deviation_value'])
pattern_p, pattern_ref = nearest_percentile_batch('sleep_pattern', scored_df['age_group_for_score'], scored_df['sleep_pattern_value'])
step_p, step_ref = nearest_percentile_batch('step', scored_df['age_group_for_score'], scored_df['step_value'])

scored_df['sleep_percentile'] = sleep_p
scored_df['sleep_ref_value'] = sleep_ref
scored_df['sleep_pattern_percentile'] = pattern_p
scored_df['sleep_pattern_ref_value'] = pattern_ref
scored_df['step_percentile'] = step_p
scored_df['step_ref_value'] = step_ref

for percentile_col, score_col in [
    ('sleep_percentile', 'sleep_score'),
    ('sleep_pattern_percentile', 'sleep_pattern_score'),
    ('step_percentile', 'step_score'),
]:
    p_values = pd.to_numeric(scored_df[percentile_col], errors='coerce').to_numpy(dtype=float)
    p_int = np.where(np.isnan(p_values), 0, p_values).astype(np.int64)
    scored_df[score_col] = np.where(np.isnan(p_values), np.nan, np.maximum(0, 100 - p_int))

scored_df['total_score'] = scored_df['sleep_score'] + scored_df['sleep_pattern_score'] + scored_df['step_score']

# threshold 캐시
threshold_cache = {}
for (domain, age_group), sub_df in score_table_valid.groupby(['domain', 'age_group_norm'], sort=False):
    pcts = pd.to_numeric(sub_df['percentile'], errors='coerce').to_numpy(dtype=float)
    vals = pd.to_numeric(sub_df['percentile_value'], errors='coerce').to_numpy(dtype=float)
    threshold_cache[(domain, age_group)] = (pcts, vals)


def _nearest_threshold_value(domain, age_group, target_percentile):
    key = (domain, age_group)
    if key not in threshold_cache:
        return np.nan
    pcts, vals = threshold_cache[key]
    if pcts.size == 0:
        return np.nan
    diff = np.abs(pcts - float(target_percentile))
    finite_mask = np.isfinite(diff)
    if not np.any(finite_mask):
        return np.nan
    finite_idx = np.flatnonzero(finite_mask)
    local_idx = int(np.argmin(diff[finite_mask]))
    return float(vals[finite_idx[local_idx]])


# 고정 타겟(30,70,70) 미리 계산
age_groups_all = pd.unique(scored_df['age_group_for_score'].dropna())
step_threshold30 = {ag: _nearest_threshold_value('step', ag, 30) for ag in age_groups_all}
sleep_dev_threshold70 = {ag: _nearest_threshold_value('sleep_time', ag, 70) for ag in age_groups_all}
sleep_pattern_threshold70 = {ag: _nearest_threshold_value('sleep_pattern', ag, 70) for ag in age_groups_all}

num_col_map = {
    'step_value': '__num_step_value',
    'DAILY_SLEEP': '__num_daily_sleep',
    'sleep_pattern_value': '__num_sleep_pattern_value',
    'sleep_deviation_value': '__num_sleep_deviation_value',
    'total_score': '__num_total_score',
}
for src_col, dst_col in num_col_map.items():
    scored_df[dst_col] = pd.to_numeric(scored_df[src_col], errors='coerce')

if 'sample_idx' in scored_df.columns:
    scored_df['__num_sample_idx'] = pd.to_numeric(scored_df['sample_idx'], errors='coerce')

group_cols_effort = ['SEQN']
if 'bootstrap_id' in scored_df.columns:
    group_cols_effort = ['bootstrap_id', 'SEQN']

metric_cols = [
    'S_imp_steps', 'S_imp_sleep_time', 'S_imp_sleep_pattern', 'S_imp',
    'A_steps', 'A_sleep_dev', 'A_sleep_pattern', 'A_sleep',
    'S_maint', 'Level_score', 'effort_score_E',
]


def _calc_effort_metrics_fast(base_df, key_cols):
    grouped = base_df.groupby(key_cols, sort=False, observed=True, group_keys=False).indices

    step_all = base_df['__num_step_value'].to_numpy(dtype=float)
    sleep_all = base_df['__num_daily_sleep'].to_numpy(dtype=float)
    pattern_all = base_df['__num_sleep_pattern_value'].to_numpy(dtype=float)
    sleep_dev_all = base_df['__num_sleep_deviation_value'].to_numpy(dtype=float)
    total_all = base_df['__num_total_score'].to_numpy(dtype=float)
    age_all = base_df['age_group_for_score'].to_numpy()

    has_sample_idx = '__num_sample_idx' in base_df.columns
    sample_all = base_df['__num_sample_idx'].to_numpy(dtype=float) if has_sample_idx else None

    records = []
    append_record = records.append

    for key, idx in grouped.items():
        key_tuple = key if isinstance(key, tuple) else (key,)
        idx_arr = np.asarray(idx, dtype=np.int64)

        if has_sample_idx:
            sample_vals = sample_all[idx_arr]
            order = np.argsort(sample_vals, kind='mergesort')
            idx_arr = idx_arr[order]
            sample_vals = sample_vals[order]

            if np.isfinite(sample_vals).any():
                current_day = int(np.nanmax(sample_vals))
            else:
                current_day = EFFORT_DEFAULT_DAY
        else:
            sample_vals = None
            current_day = EFFORT_DEFAULT_DAY

        current_start = current_day - 6
        baseline_start = current_day - 20
        baseline_end = current_day - 7
        month_start = current_day - 29

        step_vals = step_all[idx_arr]
        sleep_vals = sleep_all[idx_arr]
        pattern_vals = pattern_all[idx_arr]
        sleep_dev_vals = sleep_dev_all[idx_arr]
        total_vals = total_all[idx_arr]
        age_vals = age_all[idx_arr]

        if has_sample_idx:
            current_mask = (sample_vals >= current_start) & (sample_vals <= current_day)
            baseline_mask = (sample_vals >= baseline_start) & (sample_vals <= baseline_end)
            month_mask = (sample_vals >= month_start) & (sample_vals <= current_day)
            if not np.any(month_mask):
                month_mask = np.ones(idx_arr.size, dtype=bool)
        else:
            n = idx_arr.size
            current_mask = np.zeros(n, dtype=bool)
            current_mask[max(0, n - 7):n] = True

            baseline_mask = np.zeros(n, dtype=bool)
            baseline_mask[max(0, n - 21):max(0, n - 7)] = True

            month_mask = np.zeros(n, dtype=bool)
            month_mask[max(0, n - 30):n] = True

        cur_steps = _mean_ignore_nan(step_vals[current_mask])
        base_steps = _mean_ignore_nan(step_vals[baseline_mask])
        cur_sleep = _mean_ignore_nan(sleep_vals[current_mask])
        base_sleep = _mean_ignore_nan(sleep_vals[baseline_mask])
        cur_pattern = _mean_ignore_nan(pattern_vals[current_mask])
        base_pattern = _mean_ignore_nan(pattern_vals[baseline_mask])

        r_steps = _safe_change_ratio(cur_steps, base_steps)
        r_sleep_time = _safe_change_ratio(cur_sleep, base_sleep)
        r_sleep_pattern = _safe_change_ratio(base_pattern, cur_pattern)

        s_imp_steps = _improvement_score_from_ratio(r_steps)
        s_imp_sleep_time = _improvement_score_from_ratio(r_sleep_time)
        s_imp_sleep_pattern = _improvement_score_from_ratio(r_sleep_pattern)
        s_imp = float(np.mean([s_imp_steps, s_imp_sleep_time, s_imp_sleep_pattern]))

        month_steps = step_vals[month_mask]
        month_sleep_dev = sleep_dev_vals[month_mask]
        month_pattern = pattern_vals[month_mask]
        month_total = total_vals[month_mask]
        month_age = age_vals[month_mask]
        month_age = month_age[pd.notna(month_age)]
        age_group_value = month_age[-1] if month_age.size > 0 else np.nan

        threshold_steps = step_threshold30.get(age_group_value, np.nan)
        threshold_sleep_dev = sleep_dev_threshold70.get(age_group_value, np.nan)
        threshold_sleep_pattern = sleep_pattern_threshold70.get(age_group_value, np.nan)

        A_steps = _safe_rate_ge(month_steps, threshold_steps) if np.isfinite(threshold_steps) else np.nan
        A_sleep_dev = _safe_rate_le(month_sleep_dev, threshold_sleep_dev) if np.isfinite(threshold_sleep_dev) else np.nan
        A_sleep_pattern = _safe_rate_le(month_pattern, threshold_sleep_pattern) if np.isfinite(threshold_sleep_pattern) else np.nan

        sleep_components = [v for v in [A_sleep_dev, A_sleep_pattern] if pd.notna(v)]
        A_sleep = float(np.mean(sleep_components)) if sleep_components else np.nan

        A_steps_used = 0.0 if pd.isna(A_steps) else A_steps
        A_sleep_dev_used = 0.0 if pd.isna(A_sleep_dev) else A_sleep_dev
        A_sleep_pattern_used = 0.0 if pd.isna(A_sleep_pattern) else A_sleep_pattern
        s_maint = 100.0 * ((A_steps_used + A_sleep_dev_used + A_sleep_pattern_used) / 3.0)

        level_score = _mean_ignore_nan(month_total)
        level_score = 0.0 if pd.isna(level_score) else float(level_score)

        effort_score = (EFFORT_ALPHA * level_score + EFFORT_BETA * s_imp + EFFORT_GAMMA * s_maint) * EFFORT_RESCALE
        effort_score = float(np.clip(effort_score, 0.0, 300.0))

        record = {col: val for col, val in zip(key_cols, key_tuple)}
        record.update({
            'S_imp_steps': s_imp_steps,
            'S_imp_sleep_time': s_imp_sleep_time,
            'S_imp_sleep_pattern': s_imp_sleep_pattern,
            'S_imp': s_imp,
            'A_steps': A_steps,
            'A_sleep_dev': A_sleep_dev,
            'A_sleep_pattern': A_sleep_pattern,
            'A_sleep': A_sleep,
            'S_maint': s_maint,
            'Level_score': level_score,
            'effort_score_E': effort_score,
        })
        append_record(record)

    return pd.DataFrame.from_records(records, columns=key_cols + metric_cols)


effort_metrics = _calc_effort_metrics_fast(scored_df, group_cols_effort)

# 재실행 대비 중복 컬럼 제거
drop_cols = [col for col in metric_cols if col in scored_df.columns]
if drop_cols:
    scored_df = scored_df.drop(columns=drop_cols)

effort_metrics = effort_metrics.set_index(group_cols_effort)
if len(group_cols_effort) == 1:
    align_index = pd.Index(scored_df[group_cols_effort[0]])
else:
    align_index = pd.MultiIndex.from_frame(scored_df[group_cols_effort])

aligned_metrics = effort_metrics.reindex(align_index)
for col in metric_cols:
    scored_df[col] = aligned_metrics[col].to_numpy()

score_cols = [
    'bootstrap_id', 'bootstrap_row_id', 'SEQN', 'RIDAGEYR', 'age_group_for_score',
    'DAILY_SLEEP', 'sleep_deviation_value', 'sleep_percentile', 'sleep_score',
    'sleep_pattern_value', 'sleep_pattern_percentile', 'sleep_pattern_score',
    'avg_daily_steps', 'step_percentile', 'step_score', 'total_score',
    'Level_score', 'S_imp', 'S_maint', 'effort_score_E',
]
available_score_cols = [col for col in score_cols if col in scored_df.columns]
scored_df[available_score_cols].head(10)

,bootstrap_id,bootstrap_row_id,SEQN,RIDAGEYR,age_group_for_score,DAILY_SLEEP,sleep_deviation_value,sleep_percentile,sleep_score,sleep_pattern_value,sleep_pattern_percentile,sleep_pattern_score,avg_daily_steps,step_percentile,step_score,total_score,Level_score,S_imp,S_maint,effort_score_E
0,1,1,79005.0,44.0,40대,8.776401,0.000000,0.0,100.0,36.930408,90.0,10.0,25915.154170,0.0,100.0,210.0,146.333333,46.666667,45.555556,144.25
1,1,2,79005.0,44.0,40대,7.274750,0.000000,0.0,100.0,22.873324,90.0,10.0,12082.359375,0.0,100.0,210.0,146.333333,46.666667,45.555556,144.25
2,1,3,79005.0,44.0,40대,6.402836,0.597164,50.0,50.0,16.647151,90.0,10.0,26534.040105,0.0,100.0,160.0,146.333333,46.666667,45.555556,144.25
3,1,4,79005.0,44.0,40대,6.906218,0.093782,10.0,90.0,22.835392,90.0,10.0,3321.503304,80.0,20.0,120.0,146.333333,46.666667,45.555556,144.25
4,1,5,79005.0,44.0,40대,5.697430,1.302570,80.0,20.0,34.066591,90.0,10.0,1503.305247,90.0,10.0,40.0,146.333333,46.666667,45.555556,144.25
5,1,6,79005.0,44.0,40대,10.535909,1.535909,80.0,20.0,44.105570,90.0,10.0,5250.221201,40.0,60.0,90.0,146.333333,46.666667,45.555556,144.25
6,1,7,79005.0,44.0,40대,7.368770,0.000000,0.0,100.0,31.769998,90.0,10.0,4878.290505,50.0,50.0,160.0,146.333333,46.666667,45.555556,144.25
7,1,8,79005.0,44.0,40대,8.493799,0.000000,0.0,100.0,36.902161,90.0,10.0,20820.899310,0.0,100.0,210.0,146.333333,46.666667,45.555556,144.25
8,1,9,79005.0,44.0,40대,6.699903,0.300097,30.0,70.0,66.602476,90.0,10.0,2238.637330,90.0,10.0,90.0,146.333333,46.666667,45.555556,144.25
9,1,10,79005.0,44.0,40대,5.917801,1.082199,70.0,30.0,34.473322,90.0,10.0,5535.326805,40.0,60.0,100.0,146.333333,46.666667,45.555556,144.25


### 파라미터

In [3]:
# =========================
# 1) 입력 파라미터 (질환 4개 + 로지스틱 할인함수)
# =========================
P1 = 39380                    # 클러스터 1 기본보험료
p2 = 97580                    # 클러스터 2 기본보험료
p3 = 124990                   # 클러스터 3 기본보험료

# 로지스틱 할인함수 d(E) = L / (1 + exp(-k(E - E0)))
discount_L = 0.2              # 할인 상한 (비율)
discount_k = 0.03             # 할인 기울기
discount_E0 = 150.0           # 할인 중간점(노력 기준점)
max_discount_amount = 0.0     # 할인액 상한 (0 이하면 미적용)

# # 질환 4개(클러스터별 유병률)
# lambda1_1 = 0.62              # 심부전
# lambda1_2 = 0.84              # 관상동맥질환
# lambda1_3 = 0.59              # 협심증
# lambda1_4 = 1.08              # 심장마비
# lambda2_1 = 2.90              # 심부전
# lambda2_2 = 3.25              # 관상동맥질환
# lambda2_3 = 2.47              # 협심증
# lambda2_4 = 3.67              # 심장마비
# lambda3_1 = 7.39              # 심부전
# lambda3_2 = 8.85              # 관상동맥질환
# lambda3_3 = 5.60              # 협심증
# lambda3_4 = 8.24              # 심장마비

# alpha_1 = 2.01                # 전체 클러스터 평균 심부전
# alpha_2 = 2.3                 # 전체 클러스터 평균 관생동맥질환 
# alpha_3 = 1.71                # 전체 클러스터 평균 협심증
# alpha_4 = 2.63                # 전체 클러스터 평균 심장마비

# 질환 4개(클러스터별 유병률)
lambda1_1 = 0.62              # 심부전
lambda1_2 = 0.84              # 관상동맥질환
lambda1_3 = 0.59              # 협심증
lambda1_4 = 1.08              # 심장마비
lambda2_1 = 2.90              # 심부전
lambda2_2 = 3.25              # 관상동맥질환
lambda2_3 = 2.47              # 협심증
lambda2_4 = 3.67              # 심장마비
lambda3_1 = 7.39              # 심부전
lambda3_2 = 7.85              # 관상동맥질환
lambda3_3 = 5.60              # 협심증
lambda3_4 = 8.24              # 심장마비

alpha_1 = 2.01                # 전체 클러스터 평균 심부전
alpha_2 = 2.3                 # 전체 클러스터 평균 관생동맥질환 
alpha_3 = 1.71                # 전체 클러스터 평균 협심증
alpha_4 = 2.63                # 전체 클러스터 평균 심장마비

beta_1 = 763 / 100000 / 12    # 심부전의 월간 발생률
beta_2 = 103.5 / 100000 / 12  # 관상동맥질환의 월간 발생률
beta_3 = 107.6 / 100000 / 12  # 협심증의 월간 발생률
beta_4 = 68.2 / 100000 / 12   # 심장마비의 월간 발생률

# 노력 효과 계수
theta_step = 1.107308
theta_sleep = 0.690047
theta_sleep_pattern = 10.844074

severity = 10000000.0         # S: 사고 1건당 평균 지급보험금 (1000만원)
operating_cost = 0.0          # C: 운영비



# USE_NOTEBOOK_SCALE=True (DELTA_E_SCALE=300.0)
# data source for weighting/SRI: df_params

# [STEP]
#   raw: 2000.0 -> 7000.0
#   score: 12.041452 -> 89.982390 (delta_E=0.259803)
#   theta_step = 1.107308

# [SLEEP TIME]
#   hours: 5.00 -> 7.00
#   score: 10.000000 -> 100.000000 (delta_E=0.300000)
#   theta_sleep = 0.690047

# [SLEEP PATTERN | SRI percentile]
#   SRI p25 -> p75: 52.873563 -> 59.267241
#   delta_E=0.021312
#   theta_sleep_pattern = 10.844074

# theta(평균) = 4.213810

### MLE 최적화

In [4]:
# 노력점수 분포 적합 MLE (절단 로지스틱: 0~300 구간)
import numpy as np
import pandas as pd
from scipy.optimize import minimize, differential_evolution
from scipy.special import expit

# -----------------------------
# 0) df_params를 df와 동일 로직으로 스코어링
# -----------------------------
required_globals = ['score_table', 'age_to_group', 'sleep_deviation_from_daily_sleep', 'nearest_percentile_batch']
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise ValueError(
        f"전처리 함수/테이블이 없습니다: {missing_globals}. 먼저 df 전처리 셀을 실행하세요."
    )

required_df_params_cols = [
    'RIDAGEYR', 'DAILY_SLEEP', 'SLQ300', 'SLQ310', 'SLQ320', 'SLQ330', 'avg_daily_steps',
    'SEQN'
 ]
missing_df_params_cols = [c for c in required_df_params_cols if c not in df_params.columns]
if missing_df_params_cols:
    raise ValueError(f"df_params에 필요한 컬럼이 없습니다: {missing_df_params_cols}")

scored_df_params = df_params.copy()
scored_df_params['age_group_for_score'] = scored_df_params['RIDAGEYR'].apply(age_to_group)
scored_df_params['sleep_deviation_value'] = scored_df_params.apply(
    lambda row: sleep_deviation_from_daily_sleep(row['DAILY_SLEEP'], row['age_group_for_score']),
    axis=1,
 )
scored_df_params['sleep_pattern_value'] = (
    scored_df_params['SLQ300']
    + scored_df_params['SLQ310'].abs()
    + scored_df_params['SLQ320']
    + scored_df_params['SLQ330']
)
scored_df_params['step_value'] = scored_df_params['avg_daily_steps']

sleep_p, sleep_ref = nearest_percentile_batch(
    'sleep_time',
    scored_df_params['age_group_for_score'],
    scored_df_params['sleep_deviation_value'],
)
pattern_p, pattern_ref = nearest_percentile_batch(
    'sleep_pattern',
    scored_df_params['age_group_for_score'],
    scored_df_params['sleep_pattern_value'],
)
step_p, step_ref = nearest_percentile_batch(
    'step',
    scored_df_params['age_group_for_score'],
    scored_df_params['step_value'],
)

scored_df_params['sleep_percentile'] = sleep_p
scored_df_params['sleep_ref_value'] = sleep_ref
scored_df_params['sleep_pattern_percentile'] = pattern_p
scored_df_params['sleep_pattern_ref_value'] = pattern_ref
scored_df_params['step_percentile'] = step_p
scored_df_params['step_ref_value'] = step_ref

for percentile_col, score_col in [
    ('sleep_percentile', 'sleep_score'),
    ('sleep_pattern_percentile', 'sleep_pattern_score'),
    ('step_percentile', 'step_score'),
]:
    scored_df_params[score_col] = scored_df_params[percentile_col].apply(
        lambda p: np.nan if pd.isna(p) else max(0, 100 - int(p))
    )

scored_df_params['total_score'] = (
    scored_df_params['sleep_score']
    + scored_df_params['sleep_pattern_score']
    + scored_df_params['step_score']
)

# Cluster 컬럼 표준화
if 'Cluster' not in scored_df_params.columns:
    if 'cluster_id' in scored_df_params.columns:
        scored_df_params['Cluster'] = scored_df_params['cluster_id']
    else:
        raise ValueError("df_params(→scored_df_params)에 'Cluster' 또는 'cluster_id' 컬럼이 필요합니다.")

# -----------------------------
# 1) 월간(1~30일) 완전관측 집계 (df_params 기반)
# -----------------------------
if 'sample_idx' in scored_df_params.columns:
    monthly = (
        scored_df_params[scored_df_params['sample_idx'].between(1, 30)]
        .groupby(['SEQN', 'Cluster'], as_index=False)
        .agg(
            monthly_avg_total_score=('total_score', 'mean'),
            observed_days=('sample_idx', 'nunique')
        )
    )
    monthly = monthly[monthly['observed_days'] == 30].copy()
else:
    monthly = (
        scored_df_params
        .groupby(['SEQN', 'Cluster'], as_index=False)
        .agg(monthly_avg_total_score=('total_score', 'mean'))
    )
    monthly['observed_days'] = 30

if monthly.empty:
    raise ValueError("df_params 기반 완전관측(30일) 데이터가 없습니다. df_params를 확인하세요.")

# 2) 노력점수 벡터 준비 (df_params 기반)
E_vals = monthly['monthly_avg_total_score'].clip(0.0, 300.0).to_numpy(dtype=float)
if E_vals.size < 10:
    raise ValueError("MLE 적합을 위한 표본 수가 너무 적습니다(최소 10개 권장).")

LOW, HIGH = 0.0, 300.0
FULL_SCORE = 300.0
FIXED_FULL_SCORE_DISCOUNT_RATE = discount_L # 300점 고객 할인율을 5%로 고정
EPS = 1e-12

# 3) 절단 로지스틱 분포 음의 로그우도
#    X ~ Logistic(E0, scale=1/k), X in [0, 300] 조건부 분포
#    pdf(x) = [k*s(x)*(1-s(x))] / [S(300)-S(0)], s(x)=sigmoid(k*(x-E0))
def neg_log_likelihood(theta):
    log_k, E0 = theta
    k = float(np.exp(log_k))

    s = expit(k * (E_vals - E0))
    base_pdf = k * s * (1.0 - s)

    cdf_low = expit(k * (LOW - E0))
    cdf_high = expit(k * (HIGH - E0))
    norm_const = cdf_high - cdf_low

    if (not np.isfinite(norm_const)) or (norm_const <= EPS):
        return np.inf

    ll = np.log(np.clip(base_pdf, EPS, None)) - np.log(np.clip(norm_const, EPS, None))
    return -float(np.sum(ll))

# 4) 전역 + 로컬 최적화 (k>0 제약은 log_k로 처리)
#    k in [1e-4, 1.0], E0 in [0, 300]
bounds = [
    (np.log(1e-4), np.log(1.0)),
    (LOW, HIGH),
]

de_res = differential_evolution(neg_log_likelihood, bounds=bounds, seed=42, maxiter=200)
res = minimize(neg_log_likelihood, de_res.x, bounds=bounds, method='L-BFGS-B')
best_x = res.x if res.success else de_res.x

log_k_opt, E0_opt = [float(v) for v in best_x]
k_opt = float(np.exp(log_k_opt))

# 300점 할인율을 정확히 5%로 맞추기 위해 L 역산
# d(E)=L/(1+exp(-k(E-E0))) 이므로 d(300)=0.05가 되게 L 계산
sigmoid_full = float(expit(k_opt * (FULL_SCORE - E0_opt)))
if sigmoid_full <= EPS:
    raise ValueError("추정된 k/E0로는 300점 기준 정규화가 불안정합니다.")

L_opt = float(FIXED_FULL_SCORE_DISCOUNT_RATE / sigmoid_full)

# downstream 일관 적용 (비교 시뮬레이션은 df 기반 scored_df를 그대로 사용)
discount_L = L_opt
discount_k = k_opt
discount_E0 = E0_opt

# 5) 적합도 요약 (KS distance)
def truncated_logistic_cdf(x, k, E0, low=0.0, high=300.0):
    f_low = expit(k * (low - E0))
    f_high = expit(k * (high - E0))
    denom = np.clip(f_high - f_low, EPS, None)
    fx = expit(k * (x - E0))
    out = (fx - f_low) / denom
    return np.clip(out, 0.0, 1.0)

E_sorted = np.sort(E_vals)
n = E_sorted.size
emp_cdf = np.arange(1, n + 1, dtype=float) / n
fit_cdf = truncated_logistic_cdf(E_sorted, k_opt, E0_opt, LOW, HIGH)
ks_distance = float(np.max(np.abs(emp_cdf - fit_cdf)))

nll_opt = float(neg_log_likelihood([log_k_opt, E0_opt]))
aic = 2 * 2 + 2 * nll_opt  # 파라미터 2개(k, E0)

d300_check = float(np.clip(L_opt / (1.0 + np.exp(-k_opt * (FULL_SCORE - E0_opt))), 0.0, 1.0))

print("=== 노력점수 분포 MLE (절단 로지스틱) 결과: df_params 기반 ===")
print(f"표본 수 n = {n}")
print(f"300점 고정 할인율 = {FIXED_FULL_SCORE_DISCOUNT_RATE:.2%}")
print(f"L(역산)  = {L_opt:.6f}")
print(f"k(MLE)   = {k_opt:.6f}")
print(f"E0(MLE)  = {E0_opt:.3f}")
print(f"d(300) 확인 = {d300_check:.4%}")
print(f"NLL      = {nll_opt:.2f}")
print(f"AIC      = {aic:.2f}")
print(f"KS dist  = {ks_distance:.4f}")

monthly_sample = monthly.copy()
monthly_sample['fitted_cdf'] = truncated_logistic_cdf(
    monthly_sample['monthly_avg_total_score'].clip(LOW, HIGH).to_numpy(dtype=float),
    k_opt,
    E0_opt,
    LOW,
    HIGH,
)

monthly_sample[
    [
        'SEQN', 'Cluster', 'monthly_avg_total_score',
        'fitted_cdf'
    ]
] .head(10)

=== 노력점수 분포 MLE (절단 로지스틱) 결과: df_params 기반 ===
표본 수 n = 7382
300점 고정 할인율 = 20.00%
L(역산)  = 0.200294
k(MLE)   = 0.048319
E0(MLE)  = 165.046
d(300) 확인 = 20.0000%
NLL      = 37436.85
AIC      = 74877.69
KS dist  = 0.2196


,SEQN,Cluster,monthly_avg_total_score,fitted_cdf
0,62161.0,1,160.333333,0.443780
1,62164.0,1,154.333333,0.373744
2,62169.0,1,152.666667,0.355068
3,62172.0,1,136.333333,0.199850
4,62177.0,2,159.333333,0.431869
5,62180.0,1,147.000000,0.295040
6,62184.0,1,149.000000,0.315559
7,62189.0,1,148.666667,0.312086
8,62199.0,2,151.666667,0.344067
9,62200.0,2,153.000000,0.358771


In [5]:
# =========================
# 2) 공통 함수/데이터 준비 (Cluster 기반)
# =========================
import numpy as np
import pandas as pd

def d_logistic(E, L, k, E0):
    E_arr = np.asarray(E, dtype=float)
    return np.clip(L / (1.0 + np.exp(-k * (E_arr - E0))), 0.0, 1.0)

def delta_effort(E, E0):
    E_arr = np.asarray(E, dtype=float)
    return np.maximum(0.0, E_arr - E0) / 300.0

# Cluster 컬럼 표준화 (cluster_id 사용 데이터도 허용)
if 'Cluster' not in scored_df.columns:
    if 'cluster_id' in scored_df.columns:
        scored_df['Cluster'] = scored_df['cluster_id']
    else:
        raise ValueError("scored_df에 'Cluster' 또는 'cluster_id' 컬럼이 필요합니다.")

# =========================
# 3) seqn_monthly_score 생성 및 보험료 계산
# =========================
required_scored_cols = ['SEQN', 'Cluster', 'total_score', 'sample_idx']
missing_scored_cols = [c for c in required_scored_cols if c not in scored_df.columns]
if missing_scored_cols:
    raise ValueError(f"scored_df에 필수 컬럼이 없습니다: {missing_scored_cols}")

agg_spec = {'monthly_avg_total_score': ('total_score', 'mean'), 'observed_days': ('sample_idx', 'nunique')}
if 'Level_score' in scored_df.columns:
    agg_spec['level_score'] = ('Level_score', 'mean')
if 'S_imp' in scored_df.columns:
    agg_spec['improvement_score'] = ('S_imp', 'mean')
if 'S_maint' in scored_df.columns:
    agg_spec['maintenance_score'] = ('S_maint', 'mean')
if 'effort_score_E' in scored_df.columns:
    agg_spec['final_effort_score'] = ('effort_score_E', 'mean')

# 1~30일 완전관측 기준 월평균 점수 생성
if 'bootstrap_id' in scored_df.columns:
    monthly_by_bootstrap = (
        scored_df[scored_df['sample_idx'].between(1, 30)]
        .groupby(['bootstrap_id', 'SEQN', 'Cluster'], as_index=False)
        .agg(**agg_spec)
    )
    monthly_by_bootstrap = monthly_by_bootstrap[monthly_by_bootstrap['observed_days'] == 30].copy()

    agg_bootstrap_spec = {'monthly_avg_total_score': ('monthly_avg_total_score', 'mean')}
    for col in ['level_score', 'improvement_score', 'maintenance_score', 'final_effort_score']:
        if col in monthly_by_bootstrap.columns:
            agg_bootstrap_spec[col] = (col, 'mean')

    seqn_monthly_score = (
        monthly_by_bootstrap
        .groupby(['SEQN', 'Cluster'], as_index=False)
        .agg(**agg_bootstrap_spec)
    )
else:
    seqn_monthly_score = (
        scored_df[scored_df['sample_idx'].between(1, 30)]
        .groupby(['SEQN', 'Cluster'], as_index=False)
        .agg(**agg_spec)
    )
    seqn_monthly_score = seqn_monthly_score[seqn_monthly_score['observed_days'] == 30].copy()

if seqn_monthly_score.empty:
    raise ValueError("완전관측(30일) 데이터가 없습니다.")

# 클러스터별 기본보험료
base_premium_by_cluster = {1: P1, 2: p2, 3: p3}

# 질병별 발생률: (클러스터 유병률 / 전체 클러스터 평균 유병률) * 월간 발생률
alpha_by_disease = {1: alpha_1, 2: alpha_2, 3: alpha_3, 4: alpha_4}
beta_by_disease = {1: beta_1, 2: beta_2, 3: beta_3, 4: beta_4}
prevalence_by_cluster = {
    1: {1: lambda1_1, 2: lambda1_2, 3: lambda1_3, 4: lambda1_4},
    2: {1: lambda2_1, 2: lambda2_2, 3: lambda2_3, 4: lambda2_4},
    3: {1: lambda3_1, 2: lambda3_2, 3: lambda3_3, 4: lambda3_4},
}

def cluster_occurrence_rate(prevalence_dict):
    disease_rates = [
        (prevalence_dict[idx] / alpha_by_disease[idx]) * beta_by_disease[idx]
        for idx in alpha_by_disease
    ]
    return float(np.sum(disease_rates))

lambda0_by_cluster = {
    cluster: cluster_occurrence_rate(prevalence_dict)
    for cluster, prevalence_dict in prevalence_by_cluster.items()
}

# 노력효과 계수 통합
theta = float(np.mean([theta_step, theta_sleep, theta_sleep_pattern]))

# 최적화 결과가 있으면 우선 적용, 없으면 기본 파라미터 사용
L_used = float(globals().get('L_opt', discount_L))
k_used = float(globals().get('k_opt', discount_k))
E0_used = float(globals().get('E0_opt', discount_E0))

pricing_df = seqn_monthly_score.copy()
pricing_df['base_premium_P0'] = pricing_df['Cluster'].map(base_premium_by_cluster)
pricing_df['lambda0'] = pricing_df['Cluster'].map(lambda0_by_cluster)

if pricing_df[['base_premium_P0', 'lambda0']].isna().any().any():
    unknown_clusters = sorted(pricing_df.loc[pricing_df['base_premium_P0'].isna(), 'Cluster'].dropna().unique().tolist())
    raise ValueError(f"파라미터에 없는 Cluster 값이 있습니다: {unknown_clusters}")

if 'final_effort_score' in pricing_df.columns:
    pricing_df['effort_score_E'] = pricing_df['final_effort_score'].clip(lower=0, upper=300)
else:
    pricing_df['effort_score_E'] = pricing_df['monthly_avg_total_score'].clip(lower=0, upper=300)

pricing_df['discount_rate'] = d_logistic(pricing_df['effort_score_E'], L_used, k_used, E0_used)
pricing_df['raw_discount_amount'] = pricing_df['base_premium_P0'] * pricing_df['discount_rate']

if max_discount_amount > 0:
    pricing_df['applied_discount_amount'] = pricing_df['raw_discount_amount'].clip(upper=max_discount_amount)
else:
    pricing_df['applied_discount_amount'] = pricing_df['raw_discount_amount']

pricing_df['premium_P'] = pricing_df['base_premium_P0'] - pricing_df['applied_discount_amount']
pricing_df['delta_E'] = delta_effort(pricing_df['effort_score_E'], E0_used)
pricing_df['lambda_effective'] = pricing_df['lambda0'] * np.exp(-theta * pricing_df['delta_E'])
pricing_df['expected_loss_E_L'] = pricing_df['lambda_effective'] * severity
pricing_df['profit_Pi'] = pricing_df['premium_P'] - pricing_df['expected_loss_E_L'] - operating_cost

seqn_premium_cols = [
    'SEQN', 'Cluster', 'monthly_avg_total_score', 'level_score', 'improvement_score', 'maintenance_score',
    'effort_score_E', 'base_premium_P0', 'discount_rate', 'premium_P', 'lambda0',
    'lambda_effective', 'expected_loss_E_L', 'profit_Pi'
]
seqn_premium_cols = [c for c in seqn_premium_cols if c in pricing_df.columns]

seqn_premium = (
    pricing_df[seqn_premium_cols]
    .sort_values(['Cluster', 'SEQN'])
    .reset_index(drop=True)
)

print(f"사용 할인 파라미터: L={L_used:.4f}, k={k_used:.6f}, E0={E0_used:.2f}")
seqn_premium.head(10)

사용 할인 파라미터: L=0.2003, k=0.048319, E0=165.05


,SEQN,Cluster,monthly_avg_total_score,level_score,improvement_score,maintenance_score,effort_score_E,base_premium_P0,discount_rate,premium_P,lambda0,lambda_effective,expected_loss_E_L,profit_Pi
0,62301.0,1,150.369267,150.369267,51.148000,45.752222,148.70985,39380,0.062554,36916.610354,0.000282,0.000282,2819.037197,34097.573157
1,62371.0,1,149.447867,149.447867,51.268000,45.450889,147.91920,39380,0.060923,36980.859730,0.000282,0.000282,2819.037197,34161.822533
2,62479.0,1,149.363800,149.363800,51.052667,45.406222,147.77145,39380,0.060621,36992.761056,0.000282,0.000282,2819.037197,34173.723859
3,62771.0,1,149.232267,149.232267,51.316333,45.394000,147.74640,39380,0.060569,36994.775529,0.000282,0.000282,2819.037197,34175.738332
4,62884.0,1,149.480800,149.480800,50.739000,45.576444,147.84170,39380,0.060764,36987.106554,0.000282,0.000282,2819.037197,34168.069357
5,63261.0,1,145.527467,145.527467,51.244333,44.323556,144.46450,39380,0.054086,37250.101175,0.000282,0.000282,2819.037197,34431.063977
6,63793.0,1,150.249400,150.249400,51.216667,45.680667,148.60835,39380,0.062344,36924.910879,0.000282,0.000282,2819.037197,34105.873682
7,63882.0,1,145.731400,145.731400,50.942000,44.406222,144.56395,39380,0.054276,37242.621846,0.000282,0.000282,2819.037197,34423.584649
8,64018.0,1,149.355333,149.355333,51.146333,45.438000,147.80750,39380,0.060694,36989.860296,0.000282,0.000282,2819.037197,34170.823098
9,64133.0,1,149.541200,149.541200,51.279333,45.491333,148.01080,39380,0.061111,36973.464566,0.000282,0.000282,2819.037197,34154.427368


### 비교 시뮬레이션

In [6]:
# ==========================================
# 보험료 비교: (1) 클러스터만 vs (2) 클러스터+최종 노력점수(Effort Score)
# + bootstrap_id 단위 통계적 유의성 검정
# ==========================================
import numpy as np
import pandas as pd
from scipy import stats

# ---- 0) 입력 검증 ----
if 'Cluster' not in scored_df.columns:
    if 'cluster_id' in scored_df.columns:
        scored_df['Cluster'] = scored_df['cluster_id']
    else:
        raise ValueError("scored_df에 'Cluster' 또는 'cluster_id' 컬럼이 필요합니다.")

required_cols_compare = ['bootstrap_id', 'sample_idx', 'SEQN', 'Cluster', 'total_score']
missing_compare_cols = [c for c in required_cols_compare if c not in scored_df.columns]
if missing_compare_cols:
    raise ValueError(f"scored_df에 필수 컬럼이 없습니다: {missing_compare_cols}")

required_params = ['P1', 'p2', 'p3', 'max_discount_amount', 'discount_L', 'discount_k', 'discount_E0']
for pname in required_params:
    if pname not in globals():
        raise ValueError(f"파라미터 셀을 먼저 실행하세요. 누락: {pname}")

def d_logistic(E, L, k, E0):
    E_arr = np.asarray(E, dtype=float)
    return np.clip(L / (1.0 + np.exp(-k * (E_arr - E0))), 0.0, 1.0)

base_premium_by_cluster = {1: P1, 2: p2, 3: p3}

# ---- 1) bootstrap_id별 월간 점수(1~30일) 집계 ----
agg_spec = {'monthly_avg_total_score': ('total_score', 'mean'), 'observed_days': ('sample_idx', 'nunique')}
if 'Level_score' in scored_df.columns:
    agg_spec['level_score'] = ('Level_score', 'mean')
if 'S_imp' in scored_df.columns:
    agg_spec['improvement_score'] = ('S_imp', 'mean')
if 'S_maint' in scored_df.columns:
    agg_spec['maintenance_score'] = ('S_maint', 'mean')
if 'effort_score_E' in scored_df.columns:
    agg_spec['final_effort_score'] = ('effort_score_E', 'mean')

monthly_by_bootstrap = (
    scored_df[scored_df['sample_idx'].between(1, 30)]
    .groupby(['bootstrap_id', 'SEQN', 'Cluster'], as_index=False)
    .agg(**agg_spec)
)
monthly_by_bootstrap = monthly_by_bootstrap[monthly_by_bootstrap['observed_days'] == 30].copy()

# ---- 2) 두 방식 보험료 계산 ----
compare_df = monthly_by_bootstrap.copy()
compare_df['base_premium_P0'] = compare_df['Cluster'].map(base_premium_by_cluster)

if compare_df['base_premium_P0'].isna().any():
    unknown_clusters = sorted(compare_df.loc[compare_df['base_premium_P0'].isna(), 'Cluster'].dropna().unique().tolist())
    raise ValueError(f"기본보험료 파라미터에 없는 Cluster 값이 있습니다: {unknown_clusters}")

# 방식 A: 클러스터만 사용
compare_df['premium_cluster_only'] = compare_df['base_premium_P0']

# 방식 B: 클러스터 + 최종 노력 점수 + 로지스틱 할인
if 'final_effort_score' in compare_df.columns:
    compare_df['effort_score_E'] = compare_df['final_effort_score'].clip(lower=0, upper=300)
else:
    compare_df['effort_score_E'] = compare_df['monthly_avg_total_score'].clip(lower=0, upper=300)

compare_df['discount_rate'] = d_logistic(compare_df['effort_score_E'], discount_L, discount_k, discount_E0)
compare_df['raw_discount_amount'] = compare_df['base_premium_P0'] * compare_df['discount_rate']

if max_discount_amount > 0:
    compare_df['applied_discount_amount'] = compare_df['raw_discount_amount'].clip(upper=max_discount_amount)
else:
    compare_df['applied_discount_amount'] = compare_df['raw_discount_amount']

compare_df['premium_with_effort_score'] = compare_df['base_premium_P0'] - compare_df['applied_discount_amount']

# ---- 3) bootstrap_id 단위 요약(평균 보험료) ----
bootstrap_summary = (
    compare_df
    .groupby('bootstrap_id', as_index=False)
    .agg(
        n_seqn=('SEQN', 'nunique'),
        mean_premium_cluster_only=('premium_cluster_only', 'mean'),
        mean_premium_with_score=('premium_with_effort_score', 'mean')
    )
)
bootstrap_summary['mean_diff_score_minus_cluster'] = (
    bootstrap_summary['mean_premium_with_score'] - bootstrap_summary['mean_premium_cluster_only']
)

# ---- 4) 통계 검정 ----
diffs = bootstrap_summary['mean_diff_score_minus_cluster'].to_numpy(dtype=float)
if len(diffs) < 2:
    raise ValueError('통계 검정을 위해 bootstrap_id가 최소 2개 이상 필요합니다.')

ttest_res = stats.ttest_rel(
    bootstrap_summary['mean_premium_with_score'],
    bootstrap_summary['mean_premium_cluster_only']
)

try:
    wilcoxon_res = stats.wilcoxon(
        bootstrap_summary['mean_premium_with_score'],
        bootstrap_summary['mean_premium_cluster_only'],
        zero_method='wilcox',
        alternative='two-sided'
    )
    wilcoxon_stat = float(wilcoxon_res.statistic)
    wilcoxon_p = float(wilcoxon_res.pvalue)
except ValueError:
    wilcoxon_stat = np.nan
    wilcoxon_p = np.nan

rng = np.random.default_rng(42)
B = 5000
boot_means = np.empty(B)
for i in range(B):
    sample = rng.choice(diffs, size=len(diffs), replace=True)
    boot_means[i] = sample.mean()
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

stats_result = pd.DataFrame([
    {
        'n_bootstrap_id': int(len(diffs)),
        'mean_diff(score-cluster)': float(diffs.mean()),
        'bootstrap95ci_low': float(ci_low),
        'bootstrap95ci_high': float(ci_high),
        'paired_t_stat': float(ttest_res.statistic),
        'paired_t_pvalue': float(ttest_res.pvalue),
        'wilcoxon_stat': wilcoxon_stat,
        'wilcoxon_pvalue': wilcoxon_p,
    }
])

print('=== bootstrap_id별 평균 보험료 비교 ===')
display(bootstrap_summary)

print('\n=== 통계 검정 결과 ===')
display(stats_result)

seqn_cols = [
    'bootstrap_id', 'SEQN', 'Cluster', 'monthly_avg_total_score',
    'level_score', 'improvement_score', 'maintenance_score', 'effort_score_E',
    'premium_cluster_only', 'premium_with_effort_score'
]
seqn_cols = [c for c in seqn_cols if c in compare_df.columns]

seqn_premium_compare = compare_df[seqn_cols].sort_values(['bootstrap_id', 'Cluster', 'SEQN']).reset_index(drop=True)

print('\n=== SEQN별 보험료(일부) ===')
seqn_premium_compare.head(20)

=== bootstrap_id별 평균 보험료 비교 ===


,bootstrap_id,n_seqn,mean_premium_cluster_only,mean_premium_with_score,mean_diff_score_minus_cluster
0,1,100,62216.9,56598.507285,-5618.392715
1,2,100,62216.9,56910.095061,-5306.804939
2,3,100,62216.9,56759.592742,-5457.307258
3,4,100,62216.9,56943.361574,-5273.538426
4,5,100,62216.9,56802.366659,-5414.533341
...,...,...,...,...,...
4995,4996,100,62216.9,56667.525390,-5549.374610
4996,4997,100,62216.9,56839.740442,-5377.159558
4997,4998,100,62216.9,56759.492570,-5457.407430
4998,4999,100,62216.9,56709.197322,-5507.702678



=== 통계 검정 결과 ===


,n_bootstrap_id,mean_diff(score-cluster),bootstrap95ci_low,bootstrap95ci_high,paired_t_stat,paired_t_pvalue,wilcoxon_stat,wilcoxon_pvalue
0,5000,-5493.671581,-5496.704674,-5490.53185,-3463.366647,0.0,0.0,0.0



=== SEQN별 보험료(일부) ===


,bootstrap_id,SEQN,Cluster,monthly_avg_total_score,level_score,improvement_score,maintenance_score,effort_score_E,premium_cluster_only,premium_with_effort_score
0,1,62301.0,1,145.000000,145.000000,73.333333,42.222222,149.75,39380,36830.677386
1,1,62371.0,1,147.666667,147.666667,50.000000,45.555556,146.25,39380,37113.338788
2,1,62479.0,1,143.000000,143.000000,61.666667,38.888889,143.25,39380,37340.098853
3,1,62771.0,1,139.333333,139.333333,61.666667,38.888889,140.50,39380,37534.530977
4,1,62884.0,1,148.000000,148.000000,61.666667,46.666667,150.50,39380,36767.755590
5,1,63261.0,1,135.333333,135.333333,68.333333,40.000000,140.00,39380,37568.464891
6,1,63793.0,1,129.333333,129.333333,46.666667,38.888889,128.50,39380,38227.941364
7,1,63882.0,1,139.000000,139.000000,61.666667,43.333333,142.25,39380,37412.316787
8,1,64018.0,1,146.666667,146.666667,46.666667,45.555556,144.50,39380,37247.433222
9,1,64133.0,1,150.666667,150.666667,46.666667,44.444444,147.00,39380,37054.354192


In [7]:
# ==========================================
# 보험사 지출 예상 비교: (1) 클러스터만 vs (2) 클러스터+최종 노력점수(Effort Score)
# ==========================================
import numpy as np
import pandas as pd
from scipy import stats

# ---- 0) 입력 검증 ----
if 'Cluster' not in scored_df.columns:
    if 'cluster_id' in scored_df.columns:
        scored_df['Cluster'] = scored_df['cluster_id']
    else:
        raise ValueError("scored_df에 'Cluster' 또는 'cluster_id' 컬럼이 필요합니다.")

required_cols_spend = ['bootstrap_id', 'sample_idx', 'SEQN', 'Cluster', 'total_score']
missing_spend_cols = [c for c in required_cols_spend if c not in scored_df.columns]
if missing_spend_cols:
    raise ValueError(f"scored_df에 필수 컬럼이 없습니다: {missing_spend_cols}")

required_spend_params = [
    'theta_step', 'theta_sleep', 'theta_sleep_pattern', 'severity', 'operating_cost',
    'discount_E0',
    'alpha_1', 'alpha_2', 'alpha_3', 'alpha_4',
    'beta_1', 'beta_2', 'beta_3', 'beta_4',
    'lambda1_1', 'lambda1_2', 'lambda1_3', 'lambda1_4',
    'lambda2_1', 'lambda2_2', 'lambda2_3', 'lambda2_4',
    'lambda3_1', 'lambda3_2', 'lambda3_3', 'lambda3_4',
]
for pname in required_spend_params:
    if pname not in globals():
        raise ValueError(f"파라미터 셀을 먼저 실행하세요. 누락: {pname}")

# 도메인별 노력효과 계수 통합(평균)
theta = float(np.mean([theta_step, theta_sleep, theta_sleep_pattern]))
effort_threshold = float(np.clip(discount_E0, 0.0, 299.999))

# ---- 1) 질병별 발생률 기반 클러스터 λ0 매핑 ----
alpha_by_disease = {1: alpha_1, 2: alpha_2, 3: alpha_3, 4: alpha_4}
beta_by_disease = {1: beta_1, 2: beta_2, 3: beta_3, 4: beta_4}
prevalence_by_cluster = {
    1: {1: lambda1_1, 2: lambda1_2, 3: lambda1_3, 4: lambda1_4},
    2: {1: lambda2_1, 2: lambda2_2, 3: lambda2_3, 4: lambda2_4},
    3: {1: lambda3_1, 2: lambda3_2, 3: lambda3_3, 4: lambda3_4},
}
lambda0_by_cluster_spend = {
    cluster: float(np.sum([
        (prev_by_disease[idx] / alpha_by_disease[idx]) * beta_by_disease[idx]
        for idx in alpha_by_disease
    ]))
    for cluster, prev_by_disease in prevalence_by_cluster.items()
}

# ---- 2) bootstrap_id별 월간 점수(1~30일) 집계 ----
agg_spec = {'monthly_avg_total_score': ('total_score', 'mean'), 'observed_days': ('sample_idx', 'nunique')}
if 'Level_score' in scored_df.columns:
    agg_spec['level_score'] = ('Level_score', 'mean')
if 'S_imp' in scored_df.columns:
    agg_spec['improvement_score'] = ('S_imp', 'mean')
if 'S_maint' in scored_df.columns:
    agg_spec['maintenance_score'] = ('S_maint', 'mean')
if 'effort_score_E' in scored_df.columns:
    agg_spec['final_effort_score'] = ('effort_score_E', 'mean')

monthly_by_bootstrap_spend = (
    scored_df[scored_df['sample_idx'].between(1, 30)]
    .groupby(['bootstrap_id', 'SEQN', 'Cluster'], as_index=False)
    .agg(**agg_spec)
)
monthly_by_bootstrap_spend = monthly_by_bootstrap_spend[monthly_by_bootstrap_spend['observed_days'] == 30].copy()

# ---- 3) 두 방식 지출 계산 ----
spend_df = monthly_by_bootstrap_spend.copy()
spend_df['lambda0'] = spend_df['Cluster'].map(lambda0_by_cluster_spend)
if spend_df['lambda0'].isna().any():
    unknown_clusters = sorted(spend_df.loc[spend_df['lambda0'].isna(), 'Cluster'].dropna().unique().tolist())
    raise ValueError(f"유병률 파라미터에 없는 Cluster 값이 있습니다: {unknown_clusters}")

# 방식 A: 클러스터만 사용
spend_df['expected_expenditure_cluster_only'] = spend_df['lambda0'] * severity + operating_cost

# 방식 B: 클러스터 + 최종 노력점수 사용
if 'final_effort_score' in spend_df.columns:
    spend_df['effort_score_E'] = spend_df['final_effort_score'].clip(lower=0, upper=300)
else:
    spend_df['effort_score_E'] = spend_df['monthly_avg_total_score'].clip(lower=0, upper=300)

spend_df['delta_E'] = np.maximum(0.0, spend_df['effort_score_E'] - effort_threshold) / 300.0
spend_df['lambda_effective'] = spend_df['lambda0'] * np.exp(-theta * spend_df['delta_E'])
spend_df['expected_expenditure_with_score'] = spend_df['lambda_effective'] * severity + operating_cost

# ---- 4) bootstrap_id 단위 요약(평균 지출)
spend_bootstrap_summary = (
    spend_df
    .groupby('bootstrap_id', as_index=False)
    .agg(
        n_seqn=('SEQN', 'nunique'),
        mean_exp_cluster_only=('expected_expenditure_cluster_only', 'mean'),
        mean_exp_with_score=('expected_expenditure_with_score', 'mean')
    )
)
spend_bootstrap_summary['mean_diff_with_score_minus_cluster'] = (
    spend_bootstrap_summary['mean_exp_with_score'] - spend_bootstrap_summary['mean_exp_cluster_only']
)

# ---- 5) 통계 검정 ----
spend_diffs = spend_bootstrap_summary['mean_diff_with_score_minus_cluster'].to_numpy(dtype=float)
if len(spend_diffs) < 2:
    raise ValueError('통계 검정을 위해 bootstrap_id가 최소 2개 이상 필요합니다.')

spend_ttest_res = stats.ttest_rel(
    spend_bootstrap_summary['mean_exp_with_score'],
    spend_bootstrap_summary['mean_exp_cluster_only']
)

try:
    spend_wilcoxon_res = stats.wilcoxon(
        spend_bootstrap_summary['mean_exp_with_score'],
        spend_bootstrap_summary['mean_exp_cluster_only'],
        zero_method='wilcox',
        alternative='two-sided'
    )
    spend_wilcoxon_stat = float(spend_wilcoxon_res.statistic)
    spend_wilcoxon_p = float(spend_wilcoxon_res.pvalue)
except ValueError:
    spend_wilcoxon_stat = np.nan
    spend_wilcoxon_p = np.nan

rng = np.random.default_rng(42)
B = 5000
spend_boot_means = np.empty(B)
for i in range(B):
    sample = rng.choice(spend_diffs, size=len(spend_diffs), replace=True)
    spend_boot_means[i] = sample.mean()
spend_ci_low, spend_ci_high = np.percentile(spend_boot_means, [2.5, 97.5])

spend_stats_result = pd.DataFrame([
    {
        'n_bootstrap_id': int(len(spend_diffs)),
        'mean_diff(with_score-cluster)': float(spend_diffs.mean()),
        'bootstrap95ci_low': float(spend_ci_low),
        'bootstrap95ci_high': float(spend_ci_high),
        'paired_t_stat': float(spend_ttest_res.statistic),
        'paired_t_pvalue': float(spend_ttest_res.pvalue),
        'wilcoxon_stat': spend_wilcoxon_stat,
        'wilcoxon_pvalue': spend_wilcoxon_p,
    }
])

print('=== bootstrap_id별 보험사 예상 지출 비교 ===')
display(spend_bootstrap_summary)

print('\n=== 통계 검정 결과 ===')
display(spend_stats_result)

spend_cols = [
    'bootstrap_id', 'SEQN', 'Cluster', 'monthly_avg_total_score',
    'level_score', 'improvement_score', 'maintenance_score', 'effort_score_E',
    'expected_expenditure_cluster_only', 'expected_expenditure_with_score'
]
spend_cols = [c for c in spend_cols if c in spend_df.columns]

spend_seqn_compare = spend_df[spend_cols].sort_values(['bootstrap_id', 'Cluster', 'SEQN']).reset_index(drop=True)

print('\n=== SEQN별 예상 지출(일부) ===')
spend_seqn_compare.head(20)

=== bootstrap_id별 보험사 예상 지출 비교 ===


,bootstrap_id,n_seqn,mean_exp_cluster_only,mean_exp_with_score,mean_diff_with_score_minus_cluster
0,1,100,7870.781543,6642.819865,-1227.961677
1,2,100,7870.781543,6642.571821,-1228.209721
2,3,100,7870.781543,6631.459032,-1239.322510
3,4,100,7870.781543,6693.090632,-1177.690911
4,5,100,7870.781543,6689.507453,-1181.274089
...,...,...,...,...,...
4995,4996,100,7870.781543,6703.449826,-1167.331717
4996,4997,100,7870.781543,6653.722669,-1217.058874
4997,4998,100,7870.781543,6671.821463,-1198.960079
4998,4999,100,7870.781543,6640.779610,-1230.001933



=== 통계 검정 결과 ===


,n_bootstrap_id,mean_diff(with_score-cluster),bootstrap95ci_low,bootstrap95ci_high,paired_t_stat,paired_t_pvalue,wilcoxon_stat,wilcoxon_pvalue
0,5000,-1201.371943,-1202.502494,-1200.180534,-1998.622466,0.0,0.0,0.0



=== SEQN별 예상 지출(일부) ===


,bootstrap_id,SEQN,Cluster,monthly_avg_total_score,level_score,improvement_score,maintenance_score,effort_score_E,expected_expenditure_cluster_only,expected_expenditure_with_score
0,1,62301.0,1,145.000000,145.000000,73.333333,42.222222,149.75,2819.037197,2819.037197
1,1,62371.0,1,147.666667,147.666667,50.000000,45.555556,146.25,2819.037197,2819.037197
2,1,62479.0,1,143.000000,143.000000,61.666667,38.888889,143.25,2819.037197,2819.037197
3,1,62771.0,1,139.333333,139.333333,61.666667,38.888889,140.50,2819.037197,2819.037197
4,1,62884.0,1,148.000000,148.000000,61.666667,46.666667,150.50,2819.037197,2819.037197
5,1,63261.0,1,135.333333,135.333333,68.333333,40.000000,140.00,2819.037197,2819.037197
6,1,63793.0,1,129.333333,129.333333,46.666667,38.888889,128.50,2819.037197,2819.037197
7,1,63882.0,1,139.000000,139.000000,61.666667,43.333333,142.25,2819.037197,2819.037197
8,1,64018.0,1,146.666667,146.666667,46.666667,45.555556,144.50,2819.037197,2819.037197
9,1,64133.0,1,150.666667,150.666667,46.666667,44.444444,147.00,2819.037197,2819.037197


In [8]:
# === 클러스터별 경험적 평균 할인율 계산 (실제 데이터 기반) ===
import numpy as np
import pandas as pd

# 전제: 이전 셀에서 `compare_df`(월간 집계, discount_rate 포함)를 생성했음
if 'compare_df' not in globals():
    raise ValueError("`compare_df`가 없습니다. 먼저 '보험료 비교' 셀을 실행하세요.")

# discount 관련 컬럼이 없으면 재계산(안전장치)
if 'discount_rate' not in compare_df.columns or 'applied_discount_amount' not in compare_df.columns:
    compare_df['effort_score_E'] = compare_df['monthly_avg_total_score'].clip(lower=0, upper=300)
    compare_df['discount_rate'] = d_logistic(compare_df['effort_score_E'], discount_L, discount_k, discount_E0)
    compare_df['raw_discount_amount'] = compare_df['base_premium_P0'] * compare_df['discount_rate']
    compare_df['applied_discount_amount'] = (
        compare_df['raw_discount_amount'].clip(upper=max_discount_amount)
        if max_discount_amount > 0
        else compare_df['raw_discount_amount']
    )

# 적용 할인 비율(보험료 대비 실제 할인 비율)
compare_df['applied_discount_pct'] = compare_df['applied_discount_amount'] / compare_df['base_premium_P0']

# 클러스터별 관측치 기반 요약(경험적 평균 할인율)
cluster_emp = (
    compare_df
    .groupby('Cluster', as_index=False)
    .agg(
        n_seqn=('SEQN', 'nunique'),
        mean_discount_rate=('discount_rate', 'mean'),
        median_discount_rate=('discount_rate', 'median'),
        std_discount_rate=('discount_rate', 'std'),
        mean_applied_discount_pct=('applied_discount_pct', 'mean'),
    )
)

# 부트스트랩 단위(있으면)로 클러스터별 평균과 95% CI 계산
if 'bootstrap_id' in compare_df.columns:
    boot_cluster = (
        compare_df
        .groupby(['bootstrap_id', 'Cluster'], as_index=False)['discount_rate']
        .mean()
    )
    ci = (
        boot_cluster
        .groupby('Cluster')['discount_rate']
        .quantile([0.025, 0.975])
        .unstack(level=1)
        .rename(columns={0.025: 'ci_low', 0.975: 'ci_high'})
        .reset_index()
    )

    boot_mean = (
        boot_cluster.groupby('Cluster', as_index=False)['discount_rate'].mean()
        .rename(columns={'discount_rate': 'boot_mean_discount_rate'})
    )

    cluster_emp = (
        cluster_emp.merge(boot_mean, on='Cluster', how='left')
                   .merge(ci, on='Cluster', how='left')
    )

# 보기 좋게 퍼센트 컬럼 추가
cluster_emp['mean_discount_rate_%'] = (cluster_emp['mean_discount_rate'] * 100).round(2)
cluster_emp['median_discount_rate_%'] = (cluster_emp['median_discount_rate'] * 100).round(2)
cluster_emp['mean_applied_discount_pct_%'] = (cluster_emp['mean_applied_discount_pct'] * 100).round(2)
if 'boot_mean_discount_rate' in cluster_emp.columns:
    cluster_emp['boot_mean_discount_rate_%'] = (cluster_emp['boot_mean_discount_rate'] * 100).round(2)
    cluster_emp['ci_low_%'] = (cluster_emp['ci_low'] * 100).round(2)
    cluster_emp['ci_high_%'] = (cluster_emp['ci_high'] * 100).round(2)

# 출력
print("클러스터별 경험적 평균 할인율 (discount_rate) — 관측치 기반")
display_cols = [
    'Cluster', 'n_seqn', 'mean_discount_rate_%', 'median_discount_rate_%',
    'mean_applied_discount_pct_%'
]
if 'boot_mean_discount_rate_%' in cluster_emp.columns:
    display_cols += ['boot_mean_discount_rate_%', 'ci_low_%', 'ci_high_%']

display(cluster_emp[display_cols].sort_values('Cluster').reset_index(drop=True))

# 전체 요약(참고)
overall_mean = float(compare_df['discount_rate'].mean())
print(f"전체 관측 평균 할인율: {overall_mean:.2%}")


클러스터별 경험적 평균 할인율 (discount_rate) — 관측치 기반


,Cluster,n_seqn,mean_discount_rate_%,median_discount_rate_%,mean_applied_discount_pct_%,boot_mean_discount_rate_%,ci_low_%,ci_high_%
0,1,65,6.93,6.16,6.93,6.93,6.51,7.39
1,2,26,9.48,6.96,9.48,9.48,8.88,10.12
2,3,9,11.69,8.68,11.69,11.69,10.78,12.63


전체 관측 평균 할인율: 8.02%


In [9]:
# 진단: 할인 파라미터 및 compare_df 할인 분포 확인
import numpy as np
print('discount_L, discount_k, discount_E0 =', discount_L, discount_k, discount_E0)
print('effort_threshold(clipped) =', float(np.clip(discount_E0, 0.0, 299.999)))

if 'compare_df' in globals():
    print('\ncompare_df 요약:')
    display(compare_df[['monthly_avg_total_score', 'base_premium_P0']].describe())

    if 'discount_rate' in compare_df.columns:
        print('\ndiscount_rate 요약:')
        display(compare_df['discount_rate'].describe())
        nonzero_frac = float((compare_df['discount_rate'] > 0).mean())
        print(f"discount_rate > 0 비율: {nonzero_frac:.3%}")
        print('min,max discount_rate (%):', compare_df['discount_rate'].min()*100, compare_df['discount_rate'].max()*100)
        display(compare_df.loc[:, ['monthly_avg_total_score','effort_score_E','discount_rate','applied_discount_amount','applied_discount_pct','base_premium_P0']].head(20))
    else:
        print('compare_df에 `discount_rate` 컬럼이 없습니다.')
else:
    print('`compare_df`가 존재하지 않습니다. 먼저 보험료 비교 셀을 실행하세요.')

discount_L, discount_k, discount_E0 = 0.20029449495355373 0.04831870607399064 165.04583197569357
effort_threshold(clipped) = 165.04583197569357

compare_df 요약:


,monthly_avg_total_score,base_premium_P0
count,500000.000000,500000.000000
mean,163.162259,62216.900000
std,35.342716,31918.285119
min,101.333333,39380.000000
25%,144.000000,39380.000000
50%,151.000000,39380.000000
75%,159.666667,97580.000000
max,276.000000,124990.000000



discount_rate 요약:


count    500000.000000
mean          0.080198
std           0.047948
min           0.008882
25%           0.049967
50%           0.064208
75%           0.085031
max           0.197853
Name: discount_rate, dtype: float64

discount_rate > 0 비율: 100.000%
min,max discount_rate (%): 0.8881589391046881 19.785269590459155


,monthly_avg_total_score,effort_score_E,discount_rate,applied_discount_amount,applied_discount_pct,base_premium_P0
0,145.000000,149.75,0.064736,2549.322614,0.064736,39380
1,147.666667,146.25,0.057559,2266.661212,0.057559,39380
2,143.000000,143.25,0.051800,2039.901147,0.051800,39380
3,139.333333,140.50,0.046863,1845.469023,0.046863,39380
4,148.000000,150.50,0.066334,2612.244410,0.066334,39380
5,135.333333,140.00,0.046001,1811.535109,0.046001,39380
6,130.000000,129.00,0.029864,3732.656441,0.029864,124990
7,129.333333,128.50,0.029255,1152.058636,0.029255,39380
8,139.000000,142.25,0.049967,1967.683213,0.049967,39380
9,146.666667,144.50,0.054154,2132.566778,0.054154,39380


In [10]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# ================================================================
# [MASTER REPORT PREP] 마지막 셀 출력에 필요한 계산을 사전 준비
# ================================================================

REPORT_PERCENTILES = [0.05, 0.25, 0.5, 0.75, 0.95]


def _first_available(names, default=np.nan):
    for name in names:
        if name in globals():
            return globals()[name]
    return default


def _safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def _get_pricing_df():
    if 'pricing_df' in globals() and isinstance(pricing_df, pd.DataFrame) and not pricing_df.empty:
        return pricing_df.copy()
    if 'seqn_premium' in globals() and isinstance(seqn_premium, pd.DataFrame) and not seqn_premium.empty:
        return seqn_premium.copy()
    raise ValueError("`pricing_df` 또는 `seqn_premium` 데이터가 없습니다. 시뮬레이션 셀을 먼저 실행하세요.")


def _build_lambda0_by_cluster():
    if 'lambda0_by_cluster' in globals() and isinstance(lambda0_by_cluster, dict):
        return lambda0_by_cluster

    required = [
        'alpha_1', 'alpha_2', 'alpha_3', 'alpha_4',
        'beta_1', 'beta_2', 'beta_3', 'beta_4',
        'lambda1_1', 'lambda1_2', 'lambda1_3', 'lambda1_4',
        'lambda2_1', 'lambda2_2', 'lambda2_3', 'lambda2_4',
        'lambda3_1', 'lambda3_2', 'lambda3_3', 'lambda3_4',
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise ValueError(f"lambda0 계산에 필요한 파라미터가 없습니다: {missing}")

    alpha_by_disease = {1: alpha_1, 2: alpha_2, 3: alpha_3, 4: alpha_4}
    beta_by_disease = {1: beta_1, 2: beta_2, 3: beta_3, 4: beta_4}
    prevalence_by_cluster = {
        1: {1: lambda1_1, 2: lambda1_2, 3: lambda1_3, 4: lambda1_4},
        2: {1: lambda2_1, 2: lambda2_2, 3: lambda2_3, 4: lambda2_4},
        3: {1: lambda3_1, 2: lambda3_2, 3: lambda3_3, 4: lambda3_4},
    }

    out = {}
    for cluster, prev_by_disease in prevalence_by_cluster.items():
        rates = [
            (prev_by_disease[idx] / alpha_by_disease[idx]) * beta_by_disease[idx]
            for idx in alpha_by_disease
        ]
        out[cluster] = float(np.sum(rates))
    return out


def _describe_series(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    if s.empty:
        return {
            'count': 0,
            'mean': np.nan,
            'std': np.nan,
            'min': np.nan,
            'p05': np.nan,
            'p25': np.nan,
            'p50': np.nan,
            'p75': np.nan,
            'p95': np.nan,
            'max': np.nan,
        }

    q = s.quantile(REPORT_PERCENTILES)
    return {
        'count': int(s.shape[0]),
        'mean': float(s.mean()),
        'std': float(s.std(ddof=1)) if s.shape[0] > 1 else 0.0,
        'min': float(s.min()),
        'p05': float(q.loc[0.05]),
        'p25': float(q.loc[0.25]),
        'p50': float(q.loc[0.50]),
        'p75': float(q.loc[0.75]),
        'p95': float(q.loc[0.95]),
        'max': float(s.max()),
    }


pricing_for_report = _get_pricing_df()

# 필수 컬럼 보정(호환)
if 'effort_score_E' not in pricing_for_report.columns and 'monthly_avg_total_score' in pricing_for_report.columns:
    pricing_for_report['effort_score_E'] = pricing_for_report['monthly_avg_total_score'].clip(lower=0, upper=300)
if 'monthly_avg_total_score' not in pricing_for_report.columns and 'effort_score_E' in pricing_for_report.columns:
    pricing_for_report['monthly_avg_total_score'] = pricing_for_report['effort_score_E']

if 'expected_loss_E_L' not in pricing_for_report.columns and {'lambda_effective', 'severity'} <= set(list(globals().keys()) + list(pricing_for_report.columns)):
    sev_tmp = _safe_float(_first_available(['severity'], 10000000.0), 10000000.0)
    pricing_for_report['expected_loss_E_L'] = pricing_for_report['lambda_effective'] * sev_tmp

if 'base_premium_P0' in pricing_for_report.columns and 'premium_P' in pricing_for_report.columns:
    pricing_for_report['discount_amount'] = pricing_for_report['base_premium_P0'] - pricing_for_report['premium_P']
else:
    pricing_for_report['discount_amount'] = np.nan

if {'lambda0', 'lambda_effective'} <= set(pricing_for_report.columns):
    pricing_for_report['risk_reduction_amount'] = (pricing_for_report['lambda0'] - pricing_for_report['lambda_effective']) * _safe_float(_first_available(['severity'], 10000000.0), 10000000.0)
else:
    pricing_for_report['risk_reduction_amount'] = np.nan

# ----------------------------------------------------------------
# A) 설정/파라미터/적합도
# ----------------------------------------------------------------
sim_config = {
    'N_BOOT': int(_first_available(['N_BOOT'], pricing_for_report['SEQN'].nunique() if 'SEQN' in pricing_for_report.columns else np.nan))
}

if 'scored_df' in globals() and isinstance(scored_df, pd.DataFrame) and not scored_df.empty:
    if 'bootstrap_id' in scored_df.columns:
        sim_config['N_BOOT'] = int(scored_df['bootstrap_id'].nunique())
    if {'bootstrap_id', 'SEQN'} <= set(scored_df.columns):
        strata_by_boot = scored_df.groupby('bootstrap_id')['SEQN'].nunique()
        sim_config['total_strata_n'] = int(strata_by_boot.median())
    else:
        sim_config['total_strata_n'] = int(scored_df['SEQN'].nunique()) if 'SEQN' in scored_df.columns else np.nan

    if 'sample_idx' in scored_df.columns:
        sim_config['N_SAMPLES_PER_SEQN'] = int(scored_df['sample_idx'].nunique())
    else:
        sim_config['N_SAMPLES_PER_SEQN'] = int(_first_available(['N_SAMPLES_PER_SEQN'], 30))
else:
    sim_config['total_strata_n'] = int(_first_available(['total_strata_n'], np.nan)) if not pd.isna(_first_available(['total_strata_n'], np.nan)) else np.nan
    sim_config['N_SAMPLES_PER_SEQN'] = int(_first_available(['N_SAMPLES_PER_SEQN'], 30))

sim_config['RANDOM_SEED'] = int(_first_available(['RANDOM_SEED'], 42))

L_used_report = _safe_float(_first_available(['L_used', 'L_opt', 'discount_L'], np.nan))
k_used_report = _safe_float(_first_available(['k_used', 'k_opt', 'discount_k'], np.nan))
E0_used_report = _safe_float(_first_available(['E0_used', 'E0_opt', 'discount_E0'], np.nan))
theta_report = _safe_float(_first_available(['theta'], np.mean([
    _safe_float(_first_available(['theta_step'], 5.0), 5.0),
    _safe_float(_first_available(['theta_sleep'], 5.0), 5.0),
    _safe_float(_first_available(['theta_sleep_pattern'], 5.0), 5.0),
])), 5.0)

severity_report = _safe_float(_first_available(['severity'], 10000000.0), 10000000.0)
operating_cost_report = _safe_float(_first_available(['operating_cost'], 0.0), 0.0)

base_premium_map_report = {
    'P1': _safe_float(_first_available(['P1'], np.nan)),
    'p2': _safe_float(_first_available(['p2'], np.nan)),
    'p3': _safe_float(_first_available(['p3'], np.nan)),
}

lambda0_by_cluster_report = _build_lambda0_by_cluster()

mle_summary = {
    'k_opt': _safe_float(_first_available(['k_opt'], np.nan)),
    'E0_opt': _safe_float(_first_available(['E0_opt'], np.nan)),
    'L_opt': _safe_float(_first_available(['L_opt'], np.nan)),
    'NLL': _safe_float(_first_available(['nll_opt'], np.nan)),
    'AIC': _safe_float(_first_available(['aic'], np.nan)),
    'KS_distance': _safe_float(_first_available(['ks_distance'], np.nan)),
}

# ----------------------------------------------------------------
# B) 결과 요약/클러스터/분포/정책효과
# ----------------------------------------------------------------
portfolio_summary = {
    '총 분석 고유 개인 수': int(pricing_for_report.shape[0]),
    '평균 노력 점수': float(pd.to_numeric(pricing_for_report['monthly_avg_total_score'], errors='coerce').mean()) if 'monthly_avg_total_score' in pricing_for_report.columns else np.nan,
    '평균 할인율': float(pd.to_numeric(pricing_for_report['discount_rate'], errors='coerce').mean()) if 'discount_rate' in pricing_for_report.columns else np.nan,
    '평균 최종 보험료': float(pd.to_numeric(pricing_for_report['premium_P'], errors='coerce').mean()) if 'premium_P' in pricing_for_report.columns else np.nan,
    '평균 예상 손실': float(pd.to_numeric(pricing_for_report['expected_loss_E_L'], errors='coerce').mean()) if 'expected_loss_E_L' in pricing_for_report.columns else np.nan,
    '전체 총 예상 이익': float(pd.to_numeric(pricing_for_report['profit_Pi'], errors='coerce').sum()) if 'profit_Pi' in pricing_for_report.columns else np.nan,
    '평균 예상 이익': float(pd.to_numeric(pricing_for_report['profit_Pi'], errors='coerce').mean()) if 'profit_Pi' in pricing_for_report.columns else np.nan,
}

if 'Cluster' in pricing_for_report.columns:
    cluster_detail = (
        pricing_for_report
        .groupby('Cluster', as_index=False)
        .agg(
            개인수=('SEQN', 'count') if 'SEQN' in pricing_for_report.columns else ('Cluster', 'count'),
            평균_노력점수=('monthly_avg_total_score', 'mean') if 'monthly_avg_total_score' in pricing_for_report.columns else ('effort_score_E', 'mean'),
            평균_할인율=('discount_rate', 'mean') if 'discount_rate' in pricing_for_report.columns else ('Cluster', 'size'),
            평균_최종보험료=('premium_P', 'mean') if 'premium_P' in pricing_for_report.columns else ('Cluster', 'size'),
            평균_예상이익=('profit_Pi', 'mean') if 'profit_Pi' in pricing_for_report.columns else ('Cluster', 'size'),
            총_예상이익=('profit_Pi', 'sum') if 'profit_Pi' in pricing_for_report.columns else ('Cluster', 'size'),
        )
    )
    total_profit = cluster_detail['총_예상이익'].sum() if '총_예상이익' in cluster_detail.columns else np.nan
    if pd.notna(total_profit) and total_profit != 0:
        cluster_detail['이익기여도_%'] = cluster_detail['총_예상이익'] / total_profit * 100.0
    else:
        cluster_detail['이익기여도_%'] = np.nan
else:
    cluster_detail = pd.DataFrame()

dist_targets = [
    ('monthly_avg_total_score', '노력점수'),
    ('discount_rate', '할인율'),
    ('profit_Pi', '예상이익'),
]

overall_dist_rows = []
for col, label in dist_targets:
    if col not in pricing_for_report.columns:
        continue
    row = _describe_series(pricing_for_report[col])
    row['지표'] = label
    row['범위'] = '전체'
    overall_dist_rows.append(row)
overall_distribution = pd.DataFrame(overall_dist_rows)

cluster_dist_rows = []
if 'Cluster' in pricing_for_report.columns:
    for cluster_value, sub_df in pricing_for_report.groupby('Cluster'):
        for col, label in dist_targets:
            if col not in sub_df.columns:
                continue
            row = _describe_series(sub_df[col])
            row['지표'] = label
            row['범위'] = f'Cluster {cluster_value}'
            cluster_dist_rows.append(row)
cluster_distribution = pd.DataFrame(cluster_dist_rows)

if 'discount_rate' in pricing_for_report.columns:
    discount_applied_ratio = float((pd.to_numeric(pricing_for_report['discount_rate'], errors='coerce') > 0).mean())
else:
    discount_applied_ratio = np.nan

avg_discount_amount = float(pd.to_numeric(pricing_for_report['discount_amount'], errors='coerce').mean()) if 'discount_amount' in pricing_for_report.columns else np.nan
avg_risk_reduction_amount = float(pd.to_numeric(pricing_for_report['risk_reduction_amount'], errors='coerce').mean()) if 'risk_reduction_amount' in pricing_for_report.columns else np.nan

corr_effort_discount = np.nan
corr_effort_profit = np.nan
if {'monthly_avg_total_score', 'discount_rate'} <= set(pricing_for_report.columns):
    corr_effort_discount = float(pricing_for_report[['monthly_avg_total_score', 'discount_rate']].corr().iloc[0, 1])
if {'monthly_avg_total_score', 'profit_Pi'} <= set(pricing_for_report.columns):
    corr_effort_profit = float(pricing_for_report[['monthly_avg_total_score', 'profit_Pi']].corr().iloc[0, 1])

policy_effect_summary = pd.DataFrame([
    {
        '할인 적용 고객 비율': discount_applied_ratio,
        '평균 할인액(원)': avg_discount_amount,
        '평균 리스크 절감액(원)': avg_risk_reduction_amount,
        '노력점수-할인율 상관계수': corr_effort_discount,
        '노력점수-예상이익 상관계수': corr_effort_profit,
    }
])

# ----------------------------------------------------------------
# C-2) 고급 부트스트랩 불확실성(추가 구현)
# ----------------------------------------------------------------
bootstrap_portfolio_stats = pd.DataFrame()
bootstrap_uncertainty_summary = pd.DataFrame()
seqn_profit_boot_ci = pd.DataFrame()

can_bootstrap = (
    'scored_df' in globals()
    and isinstance(scored_df, pd.DataFrame)
    and {'bootstrap_id', 'sample_idx', 'SEQN', 'Cluster', 'total_score'} <= set(scored_df.columns)
)

if can_bootstrap:
    base_map = {
        1: _safe_float(_first_available(['P1'], np.nan)),
        2: _safe_float(_first_available(['p2'], np.nan)),
        3: _safe_float(_first_available(['p3'], np.nan)),
    }

    monthly_boot = (
        scored_df[scored_df['sample_idx'].between(1, 30)]
        .groupby(['bootstrap_id', 'SEQN', 'Cluster'], as_index=False)
        .agg(
            monthly_avg_total_score=('total_score', 'mean'),
            observed_days=('sample_idx', 'nunique')
        )
    )
    monthly_boot = monthly_boot[monthly_boot['observed_days'] == 30].copy()

    if not monthly_boot.empty:
        monthly_boot['base_premium_P0'] = monthly_boot['Cluster'].map(base_map)
        monthly_boot['lambda0'] = monthly_boot['Cluster'].map(lambda0_by_cluster_report)

        if monthly_boot[['base_premium_P0', 'lambda0']].isna().any().any():
            raise ValueError("부트스트랩 불확실성 계산 중 Cluster 파라미터 매핑 누락이 있습니다.")

        monthly_boot['effort_score_E'] = monthly_boot['monthly_avg_total_score'].clip(lower=0, upper=300)
        monthly_boot['discount_rate'] = np.clip(
            L_used_report / (1.0 + np.exp(-k_used_report * (monthly_boot['effort_score_E'] - E0_used_report))),
            0.0,
            1.0,
        )
        monthly_boot['raw_discount_amount'] = monthly_boot['base_premium_P0'] * monthly_boot['discount_rate']

        max_discount_amt = _safe_float(_first_available(['max_discount_amount'], 0.0), 0.0)
        if max_discount_amt > 0:
            monthly_boot['applied_discount_amount'] = monthly_boot['raw_discount_amount'].clip(upper=max_discount_amt)
        else:
            monthly_boot['applied_discount_amount'] = monthly_boot['raw_discount_amount']

        monthly_boot['premium_P'] = monthly_boot['base_premium_P0'] - monthly_boot['applied_discount_amount']
        monthly_boot['delta_E'] = np.maximum(0.0, monthly_boot['effort_score_E'] - E0_used_report) / 300.0
        monthly_boot['lambda_effective'] = monthly_boot['lambda0'] * np.exp(-theta_report * monthly_boot['delta_E'])
        monthly_boot['expected_loss_E_L'] = monthly_boot['lambda_effective'] * severity_report
        monthly_boot['profit_Pi'] = monthly_boot['premium_P'] - monthly_boot['expected_loss_E_L'] - operating_cost_report

        bootstrap_portfolio_stats = (
            monthly_boot
            .groupby('bootstrap_id', as_index=False)
            .agg(
                n_seqn=('SEQN', 'nunique'),
                mean_profit=('profit_Pi', 'mean'),
                mean_discount_rate=('discount_rate', 'mean'),
                mean_premium=('premium_P', 'mean'),
                mean_expected_loss=('expected_loss_E_L', 'mean'),
                total_profit=('profit_Pi', 'sum'),
            )
        )

        if not bootstrap_portfolio_stats.empty:
            ci_mean_profit = np.percentile(bootstrap_portfolio_stats['mean_profit'], [2.5, 97.5])
            ci_mean_discount = np.percentile(bootstrap_portfolio_stats['mean_discount_rate'], [2.5, 97.5])
            ci_total_profit = np.percentile(bootstrap_portfolio_stats['total_profit'], [2.5, 97.5])

            bootstrap_uncertainty_summary = pd.DataFrame([
                {
                    '지표': '부트스트랩별 포트폴리오 평균 이익',
                    '평균': float(bootstrap_portfolio_stats['mean_profit'].mean()),
                    '표준편차': float(bootstrap_portfolio_stats['mean_profit'].std(ddof=1)),
                    '95% CI 하한': float(ci_mean_profit[0]),
                    '95% CI 상한': float(ci_mean_profit[1]),
                },
                {
                    '지표': '부트스트랩별 포트폴리오 평균 할인율',
                    '평균': float(bootstrap_portfolio_stats['mean_discount_rate'].mean()),
                    '표준편차': float(bootstrap_portfolio_stats['mean_discount_rate'].std(ddof=1)),
                    '95% CI 하한': float(ci_mean_discount[0]),
                    '95% CI 상한': float(ci_mean_discount[1]),
                },
                {
                    '지표': '부트스트랩별 포트폴리오 총 이익',
                    '평균': float(bootstrap_portfolio_stats['total_profit'].mean()),
                    '표준편차': float(bootstrap_portfolio_stats['total_profit'].std(ddof=1)),
                    '95% CI 하한': float(ci_total_profit[0]),
                    '95% CI 상한': float(ci_total_profit[1]),
                },
            ])

            seqn_profit_boot_ci = (
                monthly_boot
                .groupby('SEQN')['profit_Pi']
                .agg(
                    boot_mean='mean',
                    boot_std='std',
                    q025=lambda x: np.quantile(x, 0.025),
                    q975=lambda x: np.quantile(x, 0.975),
                    n_boot='count',
                )
                .reset_index()
                .sort_values('boot_std', ascending=False)
                .reset_index(drop=True)
            )

master_report_context = {
    'sim_config': sim_config,
    'applied_params': {
        'L_used': L_used_report,
        'k_used': k_used_report,
        'E0_used': E0_used_report,
        'theta': theta_report,
        'severity': severity_report,
        'operating_cost': operating_cost_report,
        **base_premium_map_report,
    },
    'disease_params': {
        'lambdas': {
            'cluster1': [
                _safe_float(_first_available(['lambda1_1'], np.nan)),
                _safe_float(_first_available(['lambda1_2'], np.nan)),
                _safe_float(_first_available(['lambda1_3'], np.nan)),
                _safe_float(_first_available(['lambda1_4'], np.nan)),
            ],
            'cluster2': [
                _safe_float(_first_available(['lambda2_1'], np.nan)),
                _safe_float(_first_available(['lambda2_2'], np.nan)),
                _safe_float(_first_available(['lambda2_3'], np.nan)),
                _safe_float(_first_available(['lambda2_4'], np.nan)),
            ],
            'cluster3': [
                _safe_float(_first_available(['lambda3_1'], np.nan)),
                _safe_float(_first_available(['lambda3_2'], np.nan)),
                _safe_float(_first_available(['lambda3_3'], np.nan)),
                _safe_float(_first_available(['lambda3_4'], np.nan)),
            ],
        },
        'alphas': [
            _safe_float(_first_available(['alpha_1'], np.nan)),
            _safe_float(_first_available(['alpha_2'], np.nan)),
            _safe_float(_first_available(['alpha_3'], np.nan)),
            _safe_float(_first_available(['alpha_4'], np.nan)),
        ],
        'betas': [
            _safe_float(_first_available(['beta_1'], np.nan)),
            _safe_float(_first_available(['beta_2'], np.nan)),
            _safe_float(_first_available(['beta_3'], np.nan)),
            _safe_float(_first_available(['beta_4'], np.nan)),
        ],
        'lambda0_by_cluster': lambda0_by_cluster_report,
    },
    'mle_summary': mle_summary,
    'portfolio_summary': portfolio_summary,
    'cluster_detail': cluster_detail,
    'overall_distribution': overall_distribution,
    'cluster_distribution': cluster_distribution,
    'policy_effect_summary': policy_effect_summary,
    'bootstrap_portfolio_stats': bootstrap_portfolio_stats,
    'bootstrap_uncertainty_summary': bootstrap_uncertainty_summary,
    'seqn_profit_boot_ci': seqn_profit_boot_ci,
}

print("master_report_context 계산 완료: 마지막 셀에서 전체 리포트를 출력할 수 있습니다.")

master_report_context 계산 완료: 마지막 셀에서 전체 리포트를 출력할 수 있습니다.


In [11]:
# [REPORT CONTEXT ENRICHMENT] 노력점수 확장/비교 통계 컨텍스트 반영
import numpy as np
import pandas as pd

if 'master_report_context' not in globals():
    raise ValueError("`master_report_context`가 없습니다. 먼저 [MASTER REPORT PREP] 셀을 실행하세요.")

ctx = master_report_context

# 1) 노력점수 구성요소 요약
pricing_ref = None
if 'pricing_df' in globals() and isinstance(pricing_df, pd.DataFrame) and not pricing_df.empty:
    pricing_ref = pricing_df.copy()
elif 'seqn_premium' in globals() and isinstance(seqn_premium, pd.DataFrame) and not seqn_premium.empty:
    pricing_ref = seqn_premium.copy()

effort_component_summary = pd.DataFrame()
if isinstance(pricing_ref, pd.DataFrame) and not pricing_ref.empty:
    component_cols = [c for c in ['level_score', 'improvement_score', 'maintenance_score', 'effort_score_E'] if c in pricing_ref.columns]
    if component_cols:
        effort_component_summary = pricing_ref[component_cols].agg(['mean', 'std', 'min', 'median', 'max']).T.reset_index()
        effort_component_summary = effort_component_summary.rename(columns={'index': '지표', 'mean': '평균', 'std': '표준편차', 'min': '최솟값', 'median': '중앙값', 'max': '최댓값'})

effort_weights_df = pd.DataFrame([
    {
        'alpha_level': float(globals().get('EFFORT_ALPHA', 0.5)),
        'beta_improvement': float(globals().get('EFFORT_BETA', 0.2)),
        'gamma_maintenance': float(globals().get('EFFORT_GAMMA', 0.3)),
    }
])

ctx['effort_weights'] = effort_weights_df
ctx['effort_component_summary'] = effort_component_summary

# 2) 비교 시뮬레이션 결과 반영
if 'bootstrap_summary' in globals() and isinstance(bootstrap_summary, pd.DataFrame):
    ctx['premium_bootstrap_summary'] = bootstrap_summary.copy()
if 'stats_result' in globals() and isinstance(stats_result, pd.DataFrame):
    ctx['premium_stats_result'] = stats_result.copy()
if 'spend_bootstrap_summary' in globals() and isinstance(spend_bootstrap_summary, pd.DataFrame):
    ctx['spend_bootstrap_summary'] = spend_bootstrap_summary.copy()
if 'spend_stats_result' in globals() and isinstance(spend_stats_result, pd.DataFrame):
    ctx['spend_stats_result'] = spend_stats_result.copy()

# 3) 보험사 손익 비교(보험료-지출) 결과 생성/반영
if 'premium_bootstrap_summary' in ctx and 'spend_bootstrap_summary' in ctx:
    prem_df = ctx['premium_bootstrap_summary'].copy()
    spend_df_ctx = ctx['spend_bootstrap_summary'].copy()

    req_p = {'bootstrap_id', 'mean_premium_cluster_only', 'mean_premium_with_score'}
    req_s = {'bootstrap_id', 'mean_exp_cluster_only', 'mean_exp_with_score'}
    if req_p.issubset(prem_df.columns) and req_s.issubset(spend_df_ctx.columns):
        pl = prem_df[list(req_p)].merge(spend_df_ctx[list(req_s)], on='bootstrap_id', how='inner')
        if not pl.empty:
            pl['mean_profit_cluster_only'] = pl['mean_premium_cluster_only'] - pl['mean_exp_cluster_only']
            pl['mean_profit_with_score'] = pl['mean_premium_with_score'] - pl['mean_exp_with_score']
            pl['mean_diff_with_score_minus_cluster'] = pl['mean_profit_with_score'] - pl['mean_profit_cluster_only']

            diffs = pl['mean_diff_with_score_minus_cluster'].to_numpy(dtype=float)
            rng = np.random.default_rng(42)
            B = 5000
            boot_means = np.empty(B)
            for i in range(B):
                sample = rng.choice(diffs, size=len(diffs), replace=True)
                boot_means[i] = sample.mean()
            ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

            try:
                from scipy import stats
                t_res = stats.ttest_rel(pl['mean_profit_with_score'], pl['mean_profit_cluster_only'])
                try:
                    w_res = stats.wilcoxon(pl['mean_profit_with_score'], pl['mean_profit_cluster_only'], zero_method='wilcox', alternative='two-sided')
                    w_stat, w_p = float(w_res.statistic), float(w_res.pvalue)
                except ValueError:
                    w_stat, w_p = np.nan, np.nan
                t_stat, t_p = float(t_res.statistic), float(t_res.pvalue)
            except Exception:
                t_stat, t_p, w_stat, w_p = np.nan, np.nan, np.nan, np.nan

            ctx['insurer_pl_bootstrap_summary'] = pl[['bootstrap_id', 'mean_profit_cluster_only', 'mean_profit_with_score', 'mean_diff_with_score_minus_cluster']].copy()
            ctx['insurer_pl_stats_result'] = pd.DataFrame([{
                'n_bootstrap_id': int(len(diffs)),
                'mean_diff(with_score-cluster)': float(np.mean(diffs)),
                'bootstrap95ci_low': float(ci_low),
                'bootstrap95ci_high': float(ci_high),
                'paired_t_stat': t_stat,
                'paired_t_pvalue': t_p,
                'wilcoxon_stat': w_stat,
                'wilcoxon_pvalue': w_p,
            }])

master_report_context = ctx
print("master_report_context 보강 완료: 노력점수 확장/비교 통계가 반영되었습니다.")

master_report_context 보강 완료: 노력점수 확장/비교 통계가 반영되었습니다.


### 리포트

In [12]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
from scipy import stats

# =================================================================
# [FINAL MASTER REPORT - ALL IN ONE]
# =================================================================

if 'master_report_context' not in globals():
    raise ValueError("`master_report_context`가 없습니다. 바로 위의 [MASTER REPORT PREP] 셀을 먼저 실행하세요.")

ctx = master_report_context
full_report_markdown = ""


def _safe_df(value):
    return value.copy() if isinstance(value, pd.DataFrame) else pd.DataFrame()


def _safe_paired_tests(with_score_series, cluster_series):
    a = pd.to_numeric(with_score_series, errors='coerce')
    b = pd.to_numeric(cluster_series, errors='coerce')
    valid = a.notna() & b.notna()
    a = a[valid]
    b = b[valid]
    if len(a) < 2:
        return np.nan, np.nan, np.nan, np.nan

    t_res = stats.ttest_rel(a, b)
    try:
        w_res = stats.wilcoxon(a, b, zero_method='wilcox', alternative='two-sided')
        w_stat, w_p = float(w_res.statistic), float(w_res.pvalue)
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    return float(t_res.statistic), float(t_res.pvalue), w_stat, w_p


def _build_insurer_pl_from_bootstrap(premium_boot, spend_boot):
    required_premium_cols = {'bootstrap_id', 'mean_premium_cluster_only', 'mean_premium_with_score'}
    required_spend_cols = {'bootstrap_id', 'mean_exp_cluster_only', 'mean_exp_with_score'}

    if not required_premium_cols.issubset(premium_boot.columns):
        return pd.DataFrame(), pd.DataFrame()
    if not required_spend_cols.issubset(spend_boot.columns):
        return pd.DataFrame(), pd.DataFrame()

    merged = pd.merge(
        premium_boot[list(required_premium_cols)],
        spend_boot[list(required_spend_cols)],
        on='bootstrap_id',
        how='inner'
    )
    if merged.empty:
        return pd.DataFrame(), pd.DataFrame()

    merged['mean_profit_cluster_only'] = merged['mean_premium_cluster_only'] - merged['mean_exp_cluster_only']
    merged['mean_profit_with_score'] = merged['mean_premium_with_score'] - merged['mean_exp_with_score']
    merged['mean_diff_with_score_minus_cluster'] = merged['mean_profit_with_score'] - merged['mean_profit_cluster_only']

    pl_bootstrap_summary = merged[[
        'bootstrap_id',
        'mean_profit_cluster_only',
        'mean_profit_with_score',
        'mean_diff_with_score_minus_cluster',
    ]].copy()

    diffs = pl_bootstrap_summary['mean_diff_with_score_minus_cluster'].to_numpy(dtype=float)
    rng = np.random.default_rng(42)
    boot_count = 5000
    boot_means = np.empty(boot_count)
    for idx in range(boot_count):
        sample = rng.choice(diffs, size=len(diffs), replace=True)
        boot_means[idx] = sample.mean()
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

    t_stat, t_p, w_stat, w_p = _safe_paired_tests(
        pl_bootstrap_summary['mean_profit_with_score'],
        pl_bootstrap_summary['mean_profit_cluster_only']
    )

    pl_stats_result = pd.DataFrame([{
        'n_bootstrap_id': int(len(pl_bootstrap_summary)),
        'mean_diff(with_score-cluster)': float(np.mean(diffs)),
        'bootstrap95ci_low': float(ci_low),
        'bootstrap95ci_high': float(ci_high),
        'paired_t_stat': t_stat,
        'paired_t_pvalue': t_p,
        'wilcoxon_stat': w_stat,
        'wilcoxon_pvalue': w_p,
    }])

    return pl_bootstrap_summary, pl_stats_result


# --- master_report_context에서 DataFrame 변환 ---
sim_config_df = pd.DataFrame([ctx.get('sim_config', {})])
applied_params_df = pd.DataFrame([ctx.get('applied_params', {})])
mle_summary_df = pd.DataFrame([ctx.get('mle_summary', {})])
portfolio_summary_df = pd.DataFrame([ctx.get('portfolio_summary', {})])

cluster_detail_df = _safe_df(ctx.get('cluster_detail', pd.DataFrame()))
overall_dist_df = _safe_df(ctx.get('overall_distribution', pd.DataFrame()))
cluster_dist_df = _safe_df(ctx.get('cluster_distribution', pd.DataFrame()))
policy_effect_df = _safe_df(ctx.get('policy_effect_summary', pd.DataFrame()))
bootstrap_portfolio_stats_df = _safe_df(ctx.get('bootstrap_portfolio_stats', pd.DataFrame()))
bootstrap_uncertainty_df = _safe_df(ctx.get('bootstrap_uncertainty_summary', pd.DataFrame()))
seqn_profit_boot_ci_df = _safe_df(ctx.get('seqn_profit_boot_ci', pd.DataFrame()))

# --- 모델 비교 통계 ---
premium_bootstrap_summary_df = _safe_df(ctx.get('premium_bootstrap_summary', pd.DataFrame()))
premium_stats_result_df = _safe_df(ctx.get('premium_stats_result', pd.DataFrame()))
spend_bootstrap_summary_df = _safe_df(ctx.get('spend_bootstrap_summary', pd.DataFrame()))
spend_stats_result_df = _safe_df(ctx.get('spend_stats_result', pd.DataFrame()))
insurer_pl_bootstrap_summary_df = _safe_df(ctx.get('insurer_pl_bootstrap_summary', pd.DataFrame()))
insurer_pl_stats_result_df = _safe_df(ctx.get('insurer_pl_stats_result', pd.DataFrame()))

if premium_bootstrap_summary_df.empty and 'bootstrap_summary' in globals():
    premium_bootstrap_summary_df = _safe_df(globals()['bootstrap_summary'])
if premium_stats_result_df.empty and 'stats_result' in globals():
    premium_stats_result_df = _safe_df(globals()['stats_result'])
if spend_bootstrap_summary_df.empty and 'spend_bootstrap_summary' in globals():
    spend_bootstrap_summary_df = _safe_df(globals()['spend_bootstrap_summary'])
if spend_stats_result_df.empty and 'spend_stats_result' in globals():
    spend_stats_result_df = _safe_df(globals()['spend_stats_result'])

if insurer_pl_bootstrap_summary_df.empty or insurer_pl_stats_result_df.empty:
    if not premium_bootstrap_summary_df.empty and not spend_bootstrap_summary_df.empty:
        pl_summary, pl_stats = _build_insurer_pl_from_bootstrap(
            premium_bootstrap_summary_df,
            spend_bootstrap_summary_df
        )
        if insurer_pl_bootstrap_summary_df.empty:
            insurer_pl_bootstrap_summary_df = pl_summary
        if insurer_pl_stats_result_df.empty:
            insurer_pl_stats_result_df = pl_stats

# --- 노력점수 확장 부록 데이터 ---
effort_weights_df = _safe_df(ctx.get('effort_weights', pd.DataFrame()))
effort_component_summary_df = _safe_df(ctx.get('effort_component_summary', pd.DataFrame()))

# --- 질병 파라미터 ---
disease_params = ctx.get('disease_params', {})
lambda_map = disease_params.get('lambdas', {})
cluster1 = lambda_map.get('cluster1', [np.nan, np.nan, np.nan, np.nan])
cluster2 = lambda_map.get('cluster2', [np.nan, np.nan, np.nan, np.nan])
cluster3 = lambda_map.get('cluster3', [np.nan, np.nan, np.nan, np.nan])

lambda_df = pd.DataFrame([
    {'Cluster': 1, 'lambda1_1': cluster1[0], 'lambda1_2': cluster1[1], 'lambda1_3': cluster1[2], 'lambda1_4': cluster1[3]},
    {'Cluster': 2, 'lambda2_1': cluster2[0], 'lambda2_2': cluster2[1], 'lambda2_3': cluster2[2], 'lambda2_4': cluster2[3]},
    {'Cluster': 3, 'lambda3_1': cluster3[0], 'lambda3_2': cluster3[1], 'lambda3_3': cluster3[2], 'lambda3_4': cluster3[3]},
])


def _normalize_lambda_df(df):
    if {'lambda_1', 'lambda_2', 'lambda_3', 'lambda_4'}.issubset(df.columns):
        return df[['Cluster', 'lambda_1', 'lambda_2', 'lambda_3', 'lambda_4']]

    rows = []
    for _, row in df.iterrows():
        cluster = int(row['Cluster']) if not pd.isna(row['Cluster']) else None
        if cluster == 1:
            values = [row.get('lambda1_1'), row.get('lambda1_2'), row.get('lambda1_3'), row.get('lambda1_4')]
        elif cluster == 2:
            values = [row.get('lambda2_1'), row.get('lambda2_2'), row.get('lambda2_3'), row.get('lambda2_4')]
        elif cluster == 3:
            values = [row.get('lambda3_1'), row.get('lambda3_2'), row.get('lambda3_3'), row.get('lambda3_4')]
        else:
            values = [
                row.get('lambda1_1') or row.get('lambda2_1') or row.get('lambda3_1'),
                row.get('lambda1_2') or row.get('lambda2_2') or row.get('lambda3_2'),
                row.get('lambda1_3') or row.get('lambda2_3') or row.get('lambda3_3'),
                row.get('lambda1_4') or row.get('lambda2_4') or row.get('lambda3_4'),
            ]
        rows.append({
            'Cluster': cluster,
            'lambda_1': values[0],
            'lambda_2': values[1],
            'lambda_3': values[2],
            'lambda_4': values[3],
        })
    return pd.DataFrame(rows)


lambda_df = _normalize_lambda_df(lambda_df)

alphas = disease_params.get('alphas', [np.nan, np.nan, np.nan, np.nan])
betas = disease_params.get('betas', [np.nan, np.nan, np.nan, np.nan])
alphas_betas_df = pd.DataFrame([{
    'alpha_1': alphas[0], 'alpha_2': alphas[1], 'alpha_3': alphas[2], 'alpha_4': alphas[3],
    'beta_1': betas[0], 'beta_2': betas[1], 'beta_3': betas[2], 'beta_4': betas[3],
}])

lambda0_cluster_df = (
    pd.DataFrame([{'Cluster': key, 'lambda0': value} for key, value in disease_params.get('lambda0_by_cluster', {}).items()])
    .sort_values('Cluster')
    .reset_index(drop=True)
)

# 보기 형식 보정
for col in ['평균 할인율', '평균 노력 점수']:
    if col in portfolio_summary_df.columns:
        portfolio_summary_df[col] = portfolio_summary_df[col].astype(float)

if not cluster_detail_df.empty:
    numeric_cols = ['평균_노력점수', '평균_할인율', '평균_최종보험료', '평균_예상이익', '총_예상이익', '이익기여도_%']
    for col in numeric_cols:
        if col in cluster_detail_df.columns:
            cluster_detail_df[col] = pd.to_numeric(cluster_detail_df[col], errors='coerce')

if not overall_dist_df.empty:
    overall_dist_df = overall_dist_df[['범위', '지표', 'count', 'mean', 'std', 'min', 'p05', 'p25', 'p50', 'p75', 'p95', 'max']]
if not cluster_dist_df.empty:
    cluster_dist_df = cluster_dist_df[['범위', '지표', 'count', 'mean', 'std', 'min', 'p05', 'p25', 'p50', 'p75', 'p95', 'max']]

# -----------------------------------------------------------------
# 마크다운 리포트 생성
# -----------------------------------------------------------------
full_report_markdown += "# 📊 보험 시뮬레이션 결과 마스터 리포트\n\n"

full_report_markdown += "## A. 시뮬레이션 설정 및 모델 파라미터\n\n"
full_report_markdown += "### A-1) 시뮬레이션 기본 구성\n"
full_report_markdown += sim_config_df.to_string(index=False) + "\n\n"
full_report_markdown += "### A-2) 정책/모델 입력 파라미터 (최종 적용값)\n"
full_report_markdown += applied_params_df.to_string(index=False) + "\n\n"
full_report_markdown += "#### 질병 발생률 관련 파라미터 (lambdas)\n"
full_report_markdown += lambda_df.to_string(index=False) + "\n\n"
full_report_markdown += "#### 질병 발생률 관련 파라미터 (alphas, betas)\n"
full_report_markdown += alphas_betas_df.to_string(index=False) + "\n\n"
full_report_markdown += "#### 클러스터별 기준 발생률 (lambda0)\n"
full_report_markdown += lambda0_cluster_df.to_string(index=False) + "\n\n"
full_report_markdown += "### A-3) 노력 점수 분포 MLE 결과 요약\n"
full_report_markdown += mle_summary_df.to_string(index=False) + "\n\n"

full_report_markdown += "## B. 시뮬레이션 결과 요약 및 핵심 지표\n\n"
full_report_markdown += "### B-1) 전체 포트폴리오 요약 통계\n"
full_report_markdown += portfolio_summary_df.to_string(index=False) + "\n\n"
full_report_markdown += "### B-2) 클러스터(위험군)별 상세 분석\n"
if cluster_detail_df.empty:
    full_report_markdown += "클러스터 상세 분석 데이터가 없습니다.\n\n"
else:
    full_report_markdown += cluster_detail_df.sort_values('Cluster').reset_index(drop=True).to_string(index=False) + "\n\n"

full_report_markdown += "### B-3) 주요 지표 분포 분석 (전체)\n"
if overall_dist_df.empty:
    full_report_markdown += "전체 분포 통계 데이터가 없습니다.\n\n"
else:
    full_report_markdown += overall_dist_df.reset_index(drop=True).to_string(index=False) + "\n\n"

full_report_markdown += "### B-4) 주요 지표 분포 분석 (클러스터별)\n"
if cluster_dist_df.empty:
    full_report_markdown += "클러스터별 분포 통계 데이터가 없습니다.\n\n"
else:
    full_report_markdown += cluster_dist_df.sort_values(['범위', '지표']).reset_index(drop=True).to_string(index=False) + "\n\n"

full_report_markdown += "### B-5) 정책 효과 및 가치 제안\n"
full_report_markdown += policy_effect_df.to_string(index=False) + "\n\n"
full_report_markdown += """
- **할인 적용 고객 비율**이 높을수록 행동 기반 인센티브가 실제로 작동하고 있음을 의미합니다.
- **평균 할인액**은 고객 체감 혜택의 크기를 보여주며, 상품 매력도 개선 근거가 됩니다.
- **노력점수-할인율/이익 상관**은 "노력의 가치"를 정량적으로 설명합니다.
- **평균 리스크 절감액**은 행동 변화가 예상 손실 감소로 이어지는 위험관리 효과를 보여줍니다.
"""

full_report_markdown += "\n## C. 클러스터링 단독 vs 클러스터링+할인 통계 분석\n\n"
full_report_markdown += "### C-1) 보험료 비교 통계 (bootstrap 단위)\n"
if premium_bootstrap_summary_df.empty:
    full_report_markdown += "보험료 비교 요약 데이터가 없습니다. `보험료 비교` 셀을 먼저 실행하세요.\n\n"
else:
    cols = ['bootstrap_id', 'mean_premium_cluster_only', 'mean_premium_with_score', 'mean_diff_score_minus_cluster']
    cols = [col for col in cols if col in premium_bootstrap_summary_df.columns]
    full_report_markdown += premium_bootstrap_summary_df[cols].head(20).to_string(index=False) + "\n\n"

if premium_stats_result_df.empty:
    full_report_markdown += "보험료 비교 통계 검정 결과가 없습니다.\n\n"
else:
    full_report_markdown += "#### 보험료 비교 검정 결과\n"
    full_report_markdown += premium_stats_result_df.to_string(index=False) + "\n\n"

full_report_markdown += "### C-2) 보험사 손익 비교 통계 (bootstrap 단위)\n"
if insurer_pl_bootstrap_summary_df.empty:
    full_report_markdown += "보험사 손익 비교 요약 데이터가 없습니다. `보험료 비교` 및 `보험사 지출 비교` 셀을 실행하세요.\n\n"
else:
    pl_cols = ['bootstrap_id', 'mean_profit_cluster_only', 'mean_profit_with_score', 'mean_diff_with_score_minus_cluster']
    pl_cols = [col for col in pl_cols if col in insurer_pl_bootstrap_summary_df.columns]
    full_report_markdown += insurer_pl_bootstrap_summary_df[pl_cols].head(20).to_string(index=False) + "\n\n"

if insurer_pl_stats_result_df.empty:
    full_report_markdown += "보험사 손익 비교 통계 검정 결과가 없습니다.\n\n"
else:
    full_report_markdown += "#### 보험사 손익 비교 검정 결과\n"
    full_report_markdown += insurer_pl_stats_result_df.to_string(index=False) + "\n\n"

full_report_markdown += """
- `mean_diff`가 음수이면 클러스터링+할인 모델이 평균 보험료(또는 평균 손익)를 낮춘 방향입니다.
- p-value가 작을수록 두 모델 차이가 우연이 아닐 가능성이 큽니다.
- 95% 신뢰구간에 0이 포함되지 않으면 모델 차이의 방향성이 더 안정적이라고 해석할 수 있습니다.
"""

full_report_markdown += "\n## D. 부트스트래핑 활용 목적 및 추가 분석\n\n"
full_report_markdown += "### D-1) 현재 코드의 부트스트래핑 활용 목적\n"
full_report_markdown += """
1. 5000회 부트스트랩은 개인별 30일 행동 패턴의 변동성을 모사하기 위한 절차입니다.
2. 부트스트랩 결과를 평균해 `monthly_avg_total_score`를 안정적인 기대 행동 점수로 추정합니다.
3. 이를 통해 일시적 변동(노이즈)에 덜 민감한 보험료 산정 기준을 제공합니다.
"""
full_report_markdown += "\n### D-2) 고급 부트스트랩 불확실성 추정 (추가 구현)\n"
if bootstrap_uncertainty_df.empty:
    full_report_markdown += "부트스트랩 불확실성 요약을 계산하지 못했습니다. `scored_df`와 필수 파라미터를 확인하세요.\n\n"
else:
    full_report_markdown += bootstrap_uncertainty_df.to_string(index=False) + "\n\n"
    full_report_markdown += "#### 부트스트랩별 포트폴리오 통계 (일부)\n"
    full_report_markdown += bootstrap_portfolio_stats_df.head(20).to_string(index=False) + "\n\n"
    full_report_markdown += "#### 개인별 profit 부트스트랩 변동성 상위 20명\n"
    if seqn_profit_boot_ci_df.empty:
        full_report_markdown += "개인별 부트스트랩 변동성 데이터가 없습니다.\n\n"
    else:
        full_report_markdown += seqn_profit_boot_ci_df.head(20).to_string(index=False) + "\n\n"

# -----------------------------------------------------------------
# E. 노력점수 확장 반영 결과 (REPORT ADDENDUM 통합)
# -----------------------------------------------------------------
full_report_markdown += "## E. 노력 점수 확장 반영 결과\n\n"

full_report_markdown += "### E-1) 최종 노력 점수 가중치\n"
if effort_weights_df.empty:
    full_report_markdown += "가중치 정보가 없습니다.\n\n"
else:
    full_report_markdown += effort_weights_df.to_string(index=False) + "\n\n"

full_report_markdown += "### E-2) 노력 점수 구성요소 분포 (Level / Improvement / Maintenance / Effort)\n"
if effort_component_summary_df.empty:
    full_report_markdown += "구성요소 요약 데이터가 없습니다.\n\n"
else:
    full_report_markdown += effort_component_summary_df.to_string(index=False) + "\n\n"

full_report_markdown += "### E-3) 확장 점수 반영 후 비교 시뮬레이션 검정\n"
has_e3 = False

if not premium_stats_result_df.empty:
    full_report_markdown += "- 보험료 비교 통계\n"
    full_report_markdown += premium_stats_result_df.to_string(index=False) + "\n\n"
    has_e3 = True

if not spend_stats_result_df.empty:
    full_report_markdown += "- 보험사 지출 비교 통계\n"
    full_report_markdown += spend_stats_result_df.to_string(index=False) + "\n\n"
    has_e3 = True

if not insurer_pl_stats_result_df.empty:
    full_report_markdown += "- 보험사 손익 비교 통계\n"
    full_report_markdown += insurer_pl_stats_result_df.to_string(index=False) + "\n\n"
    has_e3 = True

if not has_e3:
    full_report_markdown += "비교 시뮬레이션 검정 데이터가 없습니다.\n\n"

print("리포트 출력 완료")
display(Markdown(full_report_markdown))

리포트 출력 완료


# 📊 보험 시뮬레이션 결과 마스터 리포트

## A. 시뮬레이션 설정 및 모델 파라미터

### A-1) 시뮬레이션 기본 구성
 N_BOOT  total_strata_n  N_SAMPLES_PER_SEQN  RANDOM_SEED
   5000             100                  30           42

### A-2) 정책/모델 입력 파라미터 (최종 적용값)
  L_used   k_used    E0_used   theta   severity  operating_cost      P1      p2       p3
0.200294 0.048319 165.045832 4.21381 10000000.0             0.0 39380.0 97580.0 124990.0

#### 질병 발생률 관련 파라미터 (lambdas)
 Cluster  lambda_1  lambda_2  lambda_3  lambda_4
       1      0.62      0.84      0.59      1.08
       2      2.90      3.25      2.47      3.67
       3      7.39      7.85      5.60      8.24

#### 질병 발생률 관련 파라미터 (alphas, betas)
 alpha_1  alpha_2  alpha_3  alpha_4   beta_1   beta_2  beta_3   beta_4
    2.01      2.3     1.71     2.63 0.000636 0.000086 0.00009 0.000057

#### 클러스터별 기준 발생률 (lambda0)
 Cluster  lambda0
       1 0.000282
       2 0.001248
       3 0.003104

### A-3) 노력 점수 분포 MLE 결과 요약
   k_opt     E0_opt    L_opt          NLL          AIC  KS_distance
0.048319 165.045832 0.200294 37436.847472 74877.694945      0.21959

## B. 시뮬레이션 결과 요약 및 핵심 지표

### B-1) 전체 포트폴리오 요약 통계
 총 분석 고유 개인 수   평균 노력 점수   평균 할인율    평균 최종 보험료   평균 예상 손실   전체 총 예상 이익     평균 예상 이익
          100 163.162259 0.078941 56790.483011 6669.90096 5.012058e+06 50120.582051

### B-2) 클러스터(위험군)별 상세 분석
 Cluster  개인수    평균_노력점수   평균_할인율      평균_최종보험료      평균_예상이익       총_예상이익   이익기여도_%
       1   65 154.759901 0.067772  36711.142393 33985.538060 2.209060e+06 44.074907
       2   26 174.426777 0.093888  88418.404150 77748.724918 2.021467e+06 40.332070
       3    9 191.304022 0.116422 110438.393068 86836.820361 7.815314e+05 15.593023

### B-3) 주요 지표 분포 분석 (전체)
범위   지표  count         mean          std          min          p05          p25          p50          p75          p95          max
전체 노력점수    100   163.162259    34.349657   145.249400   145.515117   149.228850   149.715167   150.105817   244.484617   244.610600
전체  할인율    100     0.078941     0.045811     0.053475     0.053977     0.060562     0.061476     0.062150     0.187268     0.187356
전체 예상이익    100 50120.582051 22274.727062 30705.971627 34096.209467 34132.434444 34179.256239 79057.754174 86458.731454 87286.133622

### B-4) 주요 지표 분포 분석 (클러스터별)
       범위   지표  count         mean         std          min          p05          p25          p50          p75          p95          max
Cluster 1 노력점수     65   154.759901   23.177130   145.331067   145.520667   149.192333   149.562933   150.017867   225.473907   244.486200
Cluster 1 예상이익     65 33985.538060  853.912098 30705.971627 31385.243426 34120.739257 34155.106060 34176.863765 34434.058533 34455.129872
Cluster 1  할인율     65     0.067772    0.030961     0.053475     0.054010     0.060541     0.061093     0.061966     0.162230     0.187328
Cluster 2 노력점수     26   174.426777   43.386984   145.249400   145.492667   149.374367   149.840700   220.786817   244.501233   244.610600
Cluster 2 예상이익     26 77748.724918 2611.386701 73552.452474 73553.115123 74926.849261 79079.831700 79184.073308 79839.462110 79881.170985
Cluster 2  할인율     26     0.093888    0.057860     0.053475     0.053903     0.060619     0.061687     0.155939     0.187265     0.187356
Cluster 3 노력점수      9   191.304022   50.440594   145.687667   147.104947   149.300933   149.877800   244.420600   244.562920   244.570733
Cluster 3 예상이익      9 86836.820361  505.648156 86179.692934 86211.202072 86380.107084 87175.776921 87282.116519 87285.583566 87286.133622
Cluster 3  할인율      9     0.116422    0.067235     0.054214     0.056630     0.060580     0.062183     0.187268     0.187284     0.187287

### B-5) 정책 효과 및 가치 제안
 할인 적용 고객 비율   평균 할인액(원)  평균 리스크 절감액(원)  노력점수-할인율 상관계수  노력점수-예상이익 상관계수
         1.0 5426.416989    1200.880582       0.999879        0.294137


- **할인 적용 고객 비율**이 높을수록 행동 기반 인센티브가 실제로 작동하고 있음을 의미합니다.
- **평균 할인액**은 고객 체감 혜택의 크기를 보여주며, 상품 매력도 개선 근거가 됩니다.
- **노력점수-할인율/이익 상관**은 "노력의 가치"를 정량적으로 설명합니다.
- **평균 리스크 절감액**은 행동 변화가 예상 손실 감소로 이어지는 위험관리 효과를 보여줍니다.

## C. 클러스터링 단독 vs 클러스터링+할인 통계 분석

### C-1) 보험료 비교 통계 (bootstrap 단위)
 bootstrap_id  mean_premium_cluster_only  mean_premium_with_score  mean_diff_score_minus_cluster
            1                    62216.9             56598.507285                   -5618.392715
            2                    62216.9             56910.095061                   -5306.804939
            3                    62216.9             56759.592742                   -5457.307258
            4                    62216.9             56943.361574                   -5273.538426
            5                    62216.9             56802.366659                   -5414.533341
            6                    62216.9             56748.857033                   -5468.042967
            7                    62216.9             56707.699584                   -5509.200416
            8                    62216.9             56762.584493                   -5454.315507
            9                    62216.9             56748.418863                   -5468.481137
           10                    62216.9             56716.735125                   -5500.164875
           11                    62216.9             56743.483283                   -5473.416717
           12                    62216.9             56749.080427                   -5467.819573
           13                    62216.9             56784.040693                   -5432.859307
           14                    62216.9             56692.300401                   -5524.599599
           15                    62216.9             56777.572504                   -5439.327496
           16                    62216.9             56696.590639                   -5520.309361
           17                    62216.9             56661.767789                   -5555.132211
           18                    62216.9             56850.106062                   -5366.793938
           19                    62216.9             56750.060421                   -5466.839579
           20                    62216.9             56720.343183                   -5496.556817

#### 보험료 비교 검정 결과
 n_bootstrap_id  mean_diff(score-cluster)  bootstrap95ci_low  bootstrap95ci_high  paired_t_stat  paired_t_pvalue  wilcoxon_stat  wilcoxon_pvalue
           5000              -5493.671581       -5496.704674         -5490.53185   -3463.366647              0.0            0.0              0.0

### C-2) 보험사 손익 비교 통계 (bootstrap 단위)
 bootstrap_id  mean_profit_cluster_only  mean_profit_with_score  mean_diff_with_score_minus_cluster
            1              54346.118457            49955.687420                        -4390.431038
            2              54346.118457            50267.523239                        -4078.595218
            3              54346.118457            50128.133710                        -4217.984748
            4              54346.118457            50250.270942                        -4095.847515
            5              54346.118457            50112.859205                        -4233.259252
            6              54346.118457            50045.704481                        -4300.413977
            7              54346.118457            50075.614596                        -4270.503861
            8              54346.118457            50091.312800                        -4254.805657
            9              54346.118457            50014.013515                        -4332.104943
           10              54346.118457            50035.720323                        -4310.398135
           11              54346.118457            50107.118273                        -4239.000184
           12              54346.118457            50147.017409                        -4199.101049
           13              54346.118457            50106.384067                        -4239.734390
           14              54346.118457            50046.136550                        -4299.981907
           15              54346.118457            50083.108689                        -4263.009769
           16              54346.118457            50045.322138                        -4300.796319
           17              54346.118457            50043.066295                        -4303.052162
           18              54346.118457            50191.888715                        -4154.229743
           19              54346.118457            50074.690467                        -4271.427991
           20              54346.118457            50041.783128                        -4304.335329

#### 보험사 손익 비교 검정 결과
 n_bootstrap_id  mean_diff(with_score-cluster)  bootstrap95ci_low  bootstrap95ci_high  paired_t_stat  paired_t_pvalue  wilcoxon_stat  wilcoxon_pvalue
           5000                   -4292.299638       -4295.329327        -4289.259044   -2754.793913              0.0            0.0              0.0


- `mean_diff`가 음수이면 클러스터링+할인 모델이 평균 보험료(또는 평균 손익)를 낮춘 방향입니다.
- p-value가 작을수록 두 모델 차이가 우연이 아닐 가능성이 큽니다.
- 95% 신뢰구간에 0이 포함되지 않으면 모델 차이의 방향성이 더 안정적이라고 해석할 수 있습니다.

## D. 부트스트래핑 활용 목적 및 추가 분석

### D-1) 현재 코드의 부트스트래핑 활용 목적

1. 5000회 부트스트랩은 개인별 30일 행동 패턴의 변동성을 모사하기 위한 절차입니다.
2. 부트스트랩 결과를 평균해 `monthly_avg_total_score`를 안정적인 기대 행동 점수로 추정합니다.
3. 이를 통해 일시적 변동(노이즈)에 덜 민감한 보험료 산정 기준을 제공합니다.

### D-2) 고급 부트스트랩 불확실성 추정 (추가 구현)
                 지표           평균         표준편차    95% CI 하한    95% CI 상한
 부트스트랩별 포트폴리오 평균 이익 5.009124e+04   109.168294 4.987427e+04 5.030764e+04
부트스트랩별 포트폴리오 평균 할인율 8.400809e-02     0.001717 8.067123e-02 8.747530e-02
  부트스트랩별 포트폴리오 총 이익 5.009124e+06 10916.829439 4.987427e+06 5.030764e+06

#### 부트스트랩별 포트폴리오 통계 (일부)
 bootstrap_id  n_seqn  mean_profit  mean_discount_rate  mean_premium  mean_expected_loss  total_profit
            1     100 49976.843432            0.085839  56325.922683         6349.079251  4.997684e+06
            2     100 50305.009802            0.082225  56669.931171         6364.921369  5.030501e+06
            3     100 50157.303731            0.083234  56536.793171         6379.489440  5.015730e+06
            4     100 50269.921617            0.080611  56651.938002         6382.016385  5.026992e+06
            5     100 50243.153841            0.081345  56596.867476         6353.713635  5.024315e+06
            6     100 50062.223315            0.083440  56466.526371         6404.303056  5.006222e+06
            7     100 50085.973552            0.085339  56435.472054         6349.498501  5.008597e+06
            8     100 50131.477693            0.084084  56494.365406         6362.887713  5.013148e+06
            9     100 50005.316123            0.084802  56404.071771         6398.755647  5.000532e+06
           10     100 50176.374650            0.083544  56549.950010         6373.575360  5.017637e+06
           11     100 50055.316659            0.084735  56439.034093         6383.717434  5.005532e+06
           12     100 50123.234556            0.083494  56447.662341         6324.427785  5.012323e+06
           13     100 50109.296541            0.083744  56460.510849         6351.214309  5.010930e+06
           14     100 50077.241208            0.084646  56418.790321         6341.549112  5.007724e+06
           15     100 50046.211992            0.085476  56426.067621         6379.855629  5.004621e+06
           16     100 50120.406411            0.084400  56455.857272         6335.450861  5.012041e+06
           17     100 50032.898722            0.084849  56372.317977         6339.419255  5.003290e+06
           18     100 50280.126185            0.079709  56631.469788         6351.343603  5.028013e+06
           19     100 50058.091973            0.084158  56451.047768         6392.955794  5.005809e+06
           20     100 50111.215112            0.083656  56480.959070         6369.743958  5.011122e+06

#### 개인별 profit 부트스트랩 변동성 상위 20명
   SEQN    boot_mean    boot_std         q025         q975  n_boot
63602.0 85883.827673 2186.425530 81724.947946 89846.924586    5000
75527.0 85738.226067 2182.228089 81650.050836 89735.172962    5000
70493.0 85715.356027 2173.759498 81684.432162 89678.393557    5000
83067.0 85857.473198 2167.239926 81724.947946 89791.349175    5000
66444.0 86711.686653 2165.779987 82052.785331 90467.968698    5000
67424.0 78735.768969 1781.243650 75203.798981 81894.427717    5000
66708.0 78670.905585 1773.337710 75203.798981 81851.039857    5000
80936.0 78730.340321 1772.722721 75160.060811 81808.279373    5000
65653.0 78759.192326 1771.139342 75203.798981 81894.427717    5000
68455.0 78610.784915 1759.659822 75182.027486 81762.855129    5000
66103.0 78604.377336 1756.312600 75160.060811 81807.182950    5000
74782.0 78647.864436 1755.679701 75225.363951 81807.182950    5000
65962.0 78659.971799 1748.403079 75225.363951 81851.039857    5000
77076.0 78748.044792 1747.884911 75203.798981 81852.124554    5000
66749.0 78641.477987 1746.421751 75182.027486 81763.963324    5000
70186.0 79512.724279 1740.963389 75652.400480 82453.829679    5000
65951.0 78692.805529 1736.883851 75203.798981 81762.855129    5000
73884.0 79473.110582 1735.634315 75652.400480 82379.278381    5000
76954.0 79436.180915 1730.627386 75573.770363 82379.278381    5000
75738.0 79450.863137 1729.142292 75652.400480 82416.771810    5000

## E. 노력 점수 확장 반영 결과

### E-1) 최종 노력 점수 가중치
 alpha_level  beta_improvement  gamma_maintenance
         0.5               0.2                0.3

### E-2) 노력 점수 구성요소 분포 (Level / Improvement / Maintenance / Effort)
               지표         평균      표준편차        최솟값        중앙값        최댓값
      level_score 163.162259 34.349657 145.249400 149.715167 244.610600
improvement_score  51.163900  0.207805  50.638333  51.160500  51.765333
maintenance_score  45.684949  1.014567  44.238889  45.574889  47.872444
   effort_score_E 158.279091 26.165367 144.143000 148.188825 220.361450

### E-3) 확장 점수 반영 후 비교 시뮬레이션 검정
- 보험료 비교 통계
 n_bootstrap_id  mean_diff(score-cluster)  bootstrap95ci_low  bootstrap95ci_high  paired_t_stat  paired_t_pvalue  wilcoxon_stat  wilcoxon_pvalue
           5000              -5493.671581       -5496.704674         -5490.53185   -3463.366647              0.0            0.0              0.0

- 보험사 지출 비교 통계
 n_bootstrap_id  mean_diff(with_score-cluster)  bootstrap95ci_low  bootstrap95ci_high  paired_t_stat  paired_t_pvalue  wilcoxon_stat  wilcoxon_pvalue
           5000                   -1201.371943       -1202.502494        -1200.180534   -1998.622466              0.0            0.0              0.0

- 보험사 손익 비교 통계
 n_bootstrap_id  mean_diff(with_score-cluster)  bootstrap95ci_low  bootstrap95ci_high  paired_t_stat  paired_t_pvalue  wilcoxon_stat  wilcoxon_pvalue
           5000                   -4292.299638       -4295.329327        -4289.259044   -2754.793913              0.0            0.0              0.0



### 부록

In [13]:
# # 로지스틱 기반 할인율 계산 (MLE/최적화 미사용)
# # 할인율 = 5% * [시그모이드(E) / 시그모이드(만점)]
# import numpy as np
# import pandas as pd

# # 1) 전체 데이터 사용: bootstrap_row_id 기준 1행=1사람
# if 'Cluster' not in scored_df.columns:
#     if 'cluster_id' in scored_df.columns:
#         scored_df['Cluster'] = scored_df['cluster_id']
#     else:
#         raise ValueError("scored_df에 'Cluster' 또는 'cluster_id' 컬럼이 필요합니다.")

# required_person_cols = ['bootstrap_row_id', 'Cluster', 'total_score']
# missing_person_cols = [c for c in required_person_cols if c not in scored_df.columns]
# if missing_person_cols:
#     raise ValueError(f"scored_df에 필수 컬럼이 없습니다: {missing_person_cols}")

# base_cols = ['bootstrap_row_id', 'Cluster', 'total_score']
# if 'SEQN' in scored_df.columns:
#     base_cols.insert(1, 'SEQN')

# monthly = scored_df[base_cols].copy()
# monthly = monthly.rename(columns={'total_score': 'monthly_avg_total_score'})
# if monthly.empty:
#     raise ValueError("집계 가능한 데이터가 없습니다. scored_df를 확인하세요.")

# # 2) 기본보험료 및 λ0 매핑
# base_map = {1: P1, 2: p2, 3: p3}
# monthly['base_premium_P0'] = monthly['Cluster'].map(base_map)

# alpha_by_disease = {1: alpha_1, 2: alpha_2, 3: alpha_3, 4: alpha_4}
# beta_by_disease = {1: beta_1, 2: beta_2, 3: beta_3, 4: beta_4}
# prevalence_by_cluster = {
#     1: {1: lambda1_1, 2: lambda1_2, 3: lambda1_3, 4: lambda1_4},
#     2: {1: lambda2_1, 2: lambda2_2, 3: lambda2_3, 4: lambda2_4},
#     3: {1: lambda3_1, 2: lambda3_2, 3: lambda3_3, 4: lambda3_4},
# }

# lambda_map = {
#     cluster: float(np.sum([
#         (prev_by_disease[idx] / alpha_by_disease[idx]) * beta_by_disease[idx]
#         for idx in alpha_by_disease
#     ]))
#     for cluster, prev_by_disease in prevalence_by_cluster.items()
# }
# monthly['lambda0'] = monthly['Cluster'].map(lambda_map)

# if monthly[['base_premium_P0', 'lambda0']].isna().any().any():
#     raise ValueError("기본보험료 또는 lambda0 매핑에 누락된 Cluster가 있습니다.")

# theta = float(np.mean([theta_step, theta_sleep, theta_sleep_pattern]))
# E_vals = monthly['monthly_avg_total_score'].clip(0, 300).to_numpy(dtype=float)
# base_vals = monthly['base_premium_P0'].to_numpy(dtype=float)
# lambda0_vals = monthly['lambda0'].to_numpy(dtype=float)

# FULL_SCORE = 300.0
# DISCOUNT_CAP = 0.05  # 만점자 할인율 5%

# # 3) 할인 함수 (고정 파라미터 사용, 최적화 없음)
# def sigmoid_score(E, k, E0):
#     E_arr = np.asarray(E, dtype=float)
#     return 1.0 / (1.0 + np.exp(-k * (E_arr - E0)))

# # 파라미터 셀의 로지스틱 파라미터를 그대로 사용 (k, E0)
# k_used = float(discount_k)
# E0_used = float(discount_E0)

# sigmoid_vals = sigmoid_score(E_vals, k_used, E0_used)
# sigmoid_full = float(sigmoid_score(np.array([FULL_SCORE]), k_used, E0_used)[0])

# if sigmoid_full <= 0:
#     raise ValueError("만점 시그모이드 값이 0 이하입니다. discount_k/discount_E0를 확인하세요.")

# # 요청식: cap * (해당 점수 로지스틱/시그모이드 값) / 만점 로지스틱/시그모이드 값
# d_opt = DISCOUNT_CAP * (sigmoid_vals / sigmoid_full)
# d_opt = np.clip(d_opt, 0.0, DISCOUNT_CAP)

# raw_discount_opt = base_vals * d_opt
# if max_discount_amount > 0:
#     applied_discount_opt = np.minimum(raw_discount_opt, max_discount_amount)
# else:
#     applied_discount_opt = raw_discount_opt

# premium_opt = base_vals - applied_discount_opt
# delta_E_opt = np.maximum(0.0, E_vals - E0_used) / 300.0
# lambda_eff_opt = lambda0_vals * np.exp(-theta * delta_E_opt)
# expected_loss_opt = lambda_eff_opt * severity + operating_cost
# profit_opt = premium_opt - expected_loss_opt

# full_score_mask = np.isclose(E_vals, FULL_SCORE)
# full_score_discount_mean = float(np.mean(d_opt[full_score_mask])) if np.any(full_score_mask) else np.nan

# print("=== 로지스틱 비율 기반 할인 결과 (MLE 미사용) ===")
# print("집계 기준: bootstrap_row_id 기준 1행=1사람(모든 행 독립)")
# print(f"사용 데이터 행 수 = {len(monthly):,}")
# print(f"사용 파라미터: k={k_used:.6f}, E0={E0_used:.3f}")
# print(f"할인율 식: d(E) = {DISCOUNT_CAP:.2%} * sigmoid(E) / sigmoid(300)")
# print(f"만점자 수(E=300) = {int(full_score_mask.sum())}")
# if np.any(full_score_mask):
#     print(f"만점자 평균 할인율 = {full_score_discount_mean:.2%}")
# else:
#     print("만점자 데이터가 없어 만점 할인율 실측값은 계산되지 않았습니다.")
# print(f"평균 이익(보험사) = {float(np.mean(profit_opt)):.2f}")
# print(f"평균 보험료(고객) = {float(np.mean(premium_opt)):.2f}")
# print(f"평균 할인비율    = {float(np.mean(applied_discount_opt / base_vals)):.4f}")

# monthly_sample = monthly.copy()
# monthly_sample['discount_rate'] = d_opt
# monthly_sample['applied_discount_amount'] = applied_discount_opt
# monthly_sample['premium_with_score'] = premium_opt
# monthly_sample['delta_E'] = delta_E_opt
# monthly_sample['lambda_effective'] = lambda_eff_opt
# monthly_sample['expected_loss'] = expected_loss_opt
# monthly_sample['profit'] = profit_opt

# id_cols = ['bootstrap_row_id']
# if 'SEQN' in monthly_sample.columns:
#     id_cols.append('SEQN')
# id_cols.append('Cluster')

# monthly_sample[
#     id_cols + [
#         'monthly_avg_total_score', 'base_premium_P0',
#         'discount_rate', 'applied_discount_amount', 'premium_with_score',
#         'lambda0', 'lambda_effective', 'expected_loss', 'profit'
#     ]
# ].head(500)

In [14]:
import numpy as np
import pandas as pd                                                                                                  
import re                                                                                                            
                                                                                                                    
# =========================================================                                                          
# 통합 셀: Step + Sleep Time + Sleep Pattern(SRI percentile) theta 계산                                              
# - Step/Sleep Time: score_table 기반 연령가중 점수 매핑                                                             
# - Sleep Pattern: df_params에서 개인별 SRI 계산 후 백분위수(P25->P75)로 theta 산출                                  
# =========================================================                                                          
                                                                                                                    
# -------------------------                                                                                          
# 0) 설정                                                                                                            
# -------------------------                                                                                          
USE_NOTEBOOK_SCALE = bool(globals().get("USE_NOTEBOOK_SCALE", True))                                                 
DELTA_E_SCALE = float(globals().get("DELTA_E_SCALE", 300.0 if USE_NOTEBOOK_SCALE else 100.0))                        
                                                                                                                    
rr_step = float(globals().get("rr_step", 0.75))
rr_sleep = float(globals().get("rr_sleep", 1 / 1.23))                                                                
rr_sleep_pattern = float(globals().get("rr_sleep_pattern", 1 / 1.26))                                                
                                                                                                                    
# Step 시나리오                                                                                                      
step_raw_baseline = 2000.0                                                                                           
step_raw_post = 7000.0                                                                                               
                                                                                                                    
# Sleep Time 시나리오 (시간)                                                                                         
sleep_hours_baseline = 5.0                                                                                           
sleep_hours_post = 7.0                                                                                               
                                                                                                                    
# Sleep Pattern(SRI) 백분위수 시나리오                                                                               
SRI_BASELINE_PERCENTILE = 25.0                                                                                       
SRI_POST_PERCENTILE = 75.0                                                                                           
                                                                                                                    
# SRI 계산 설정                                                                                                      
EPOCH_MIN = 30                                                                                                       
WEEKEND_DOW = {6, 7}  # sample_idx=1을 월요일로 가정                                                                 
                                                                                                                    
# -------------------------                                                                                          
# 1) 사전 검증                                                                                                       
# -------------------------                                                                                          
for needed in ["df_params", "score_table", "age_to_group", "sleep_deviation_from_daily_sleep"]:                      
    if needed not in globals():                                                                                      
        raise ValueError(f"{needed}가 없습니다. 관련 전처리 셀을 먼저 실행하세요.")                                  
                                                                                                                    
required_df_cols = ["SEQN", "RIDAGEYR", "sample_idx", "SLQ300", "SLQ310", "SLQ320", "SLQ330", "DAILY_SLEEP",         
"avg_daily_steps"]
missing_df_cols = [c for c in required_df_cols if c not in df_params.columns]                                        
if missing_df_cols:                                                                                                  
    raise ValueError(f"df_params에 필요한 컬럼이 없습니다: {missing_df_cols}")                                       
                                                                                                                    
for name, rr in [("rr_step", rr_step), ("rr_sleep", rr_sleep), ("rr_sleep_pattern", rr_sleep_pattern)]:              
    if not (0.0 < rr < 1.0):                                                                                         
        raise ValueError(f"{name}는 (0,1) 범위여야 합니다. 현재값: {rr}")                                            

if not (0 <= SRI_BASELINE_PERCENTILE < SRI_POST_PERCENTILE <= 100):                                                  
    raise ValueError("SRI 백분위수는 0~100 범위에서 baseline < post 여야 합니다.")                                   
                                                                                                                    
# -------------------------
# 2) score_table 준비                                                                                                
# -------------------------                                                                                          
st = score_table.copy()                                                                                              
if "age_group_norm" not in st.columns:                                                                               
    if "normalize_age_group" in globals():                                                                           
        st["age_group_norm"] = st["age_group"].apply(normalize_age_group)                                            
    else:                                                                                                            
        st["age_group_norm"] = st["age_group"].astype(str).str.strip()

st = st[st["domain"].isin(["step", "sleep_time", "sleep_pattern"])].copy()
st["percentile"] = pd.to_numeric(st["percentile"], errors="coerce")                                                  
st["percentile_value"] = pd.to_numeric(st["percentile_value"], errors="coerce")
st = st.dropna(subset=["domain", "age_group_norm", "percentile", "percentile_value"])                                
                                                                                                                    
# 연령 가중치: df_params 기준                                                                                        
age_groups = df_params["RIDAGEYR"].apply(age_to_group)                                                               
age_weights = age_groups.value_counts(normalize=True, dropna=True)                                                   
                                                                                                                    
def nearest_score(domain, age_group, raw_value):                                                                     
    sub = st[(st["domain"] == domain) & (st["age_group_norm"] == age_group)].copy()                                  
    if sub.empty:                                                                                                    
        return np.nan, np.nan, np.nan                                                                                
    sub = sub.sort_values("percentile_value", kind="mergesort")                                                      
    vals = sub["percentile_value"].to_numpy(dtype=float)                                                             
    pcts = sub["percentile"].to_numpy(dtype=float)                                                                   
                                                                                                                    
    x = float(raw_value)                                                                                             
    ins = np.searchsorted(vals, x, side="left")                                                                      
    r = int(np.clip(ins, 0, len(vals) - 1))                                                                          
    l = int(np.clip(ins - 1, 0, len(vals) - 1))                                                                      
    idx = r if abs(x - vals[r]) < abs(x - vals[l]) else l                                                            
                                                                                                                    
    pct = float(pcts[idx])                                                                                           
    score = max(0.0, 100.0 - float(int(pct)))                                                                        
    ref = float(vals[idx])                                                                                           
    return score, pct, ref                                                                                           
                                                                                                                    
def weighted_scores_for_domain(domain, raw_baseline_by_age, raw_post_by_age):                                        
    rows = []                                                                                                        
    for ag, w in age_weights.items():                                                                                
        rb = raw_baseline_by_age(ag)                                                                                 
        rp = raw_post_by_age(ag)                                                                                     
                                                                                                                    
        sb, pb, rb_ref = nearest_score(domain, ag, rb)                                                               
        sp, pp, rp_ref = nearest_score(domain, ag, rp)                                                               
                                                                                                                    
        if np.isfinite(sb) and np.isfinite(sp):                                                                      
            rows.append({                                                                                            
                "age_group": ag,                                                                                     
                "weight": float(w),                                                                                  
                "raw_baseline": float(rb),                                                                           
                "raw_post": float(rp),                                                                               
                "score_baseline": float(sb),                                                                         
                "score_post": float(sp),                                                                             
                "pct_baseline": float(pb),                                                                           
                "pct_post": float(pp),                                                                               
                "ref_baseline": float(rb_ref),                                                                       
                "ref_post": float(rp_ref),                                                                           
            })                                                                                                       
                                                                                                                    
    detail = pd.DataFrame(rows)                                                                                      
    if detail.empty:
        raise ValueError(f"{domain}: 연령가중 점수 매핑 결과가 비어 있습니다.")                                      
    detail["weight"] = detail["weight"] / detail["weight"].sum()                                                     
                                                                                                                    
    score_baseline = float((detail["score_baseline"] * detail["weight"]).sum())                                      
    score_post = float((detail["score_post"] * detail["weight"]).sum())                                              
    return score_baseline, score_post, detail                                                                        
                                                                                                                    
def calc_theta_from_rr(score_baseline, score_post, rr, scale):                                                       
    delta_e = (float(score_post) - float(score_baseline)) / float(scale)                                             
    if delta_e <= 0:                                                                                                 
        raise ValueError(                                                                                            
            f"delta_E가 0 이하입니다. baseline={score_baseline:.4f}, post={score_post:.4f}, delta_E={delta_e:.6f}"   
        )                                                                                                            
    return float(-np.log(float(rr)) / delta_e), float(delta_e)                                                       

# -------------------------
# 3) Step theta                                                                                                      
# -------------------------                                                                                          
step_score_baseline, step_score_post, step_detail = weighted_scores_for_domain(                                      
    "step",                                                                                                          
    raw_baseline_by_age=lambda ag: step_raw_baseline,                                                                
    raw_post_by_age=lambda ag: step_raw_post,                                                                        
)                                                                                                                    
theta_step, delta_e_step = calc_theta_from_rr(step_score_baseline, step_score_post, rr_step, DELTA_E_SCALE)          
                                                                                                                    
# -------------------------                                                                                          
# 4) Sleep Time theta                                                                                                
# -------------------------                                                                                          
sleep_score_baseline, sleep_score_post, sleep_detail = weighted_scores_for_domain(                                   
    "sleep_time",                                                                                                    
    raw_baseline_by_age=lambda ag: sleep_deviation_from_daily_sleep(sleep_hours_baseline, ag),                       
    raw_post_by_age=lambda ag: sleep_deviation_from_daily_sleep(sleep_hours_post, ag),                               
)                                                                                                                    
theta_sleep, delta_e_sleep = calc_theta_from_rr(sleep_score_baseline, sleep_score_post, rr_sleep, DELTA_E_SCALE)     
                                                                                                                    
# -------------------------                                                                                          
# 5) Sleep Pattern theta (SRI percentile 사용)                                                                       
# -------------------------                                                                                          
def parse_time_series_to_hour(s: pd.Series) -> pd.Series:                                                            
    num = pd.to_numeric(s, errors="coerce")                                                                          
    out = num % 24.0                                                                                                 
    miss = out.isna()                                                                                                
    if not miss.any():                                                                                               
        return out                                                                                                   
                                                                                                                    
    ss = s[miss].astype(str).str.strip()                                                                             
    ss = ss.str.replace("오전", " AM ", regex=False).str.replace("오후", " PM ", regex=False)                        
    ss = ss.str.upper()                                                                                              
    cleaned = ss.str.replace(r"[^0-9:APM ]", "", regex=True)                                                         
                                                                                                                    
    def _parse_one(x: str):                                                                                          
        x = x.strip()                                                                                                
        if x == "":                                                                                                  
            return np.nan                                                                                            
        is_pm = "PM" in x                                                                                            
        is_am = "AM" in x                                                                                            
        x = re.sub(r"[^0-9:]", "", x)                                                                                
        if x == "":                                                                                                  
            return np.nan                                                                                            
        try:                                                                                                         
            if ":" in x:                                                                                             
                h, m = x.split(":")                                                                                  
                h, m = int(h), int(m)                                                                                
            elif len(x) <= 2:                                                                                        
                h, m = int(x), 0
            else:                                                                                                    
                h, m = int(x[:-2]), int(x[-2:])                                                                      
            if is_pm and h < 12:                                                                                     
                h += 12                                                                                              
            if is_am and h == 12:                                                                                    
                h = 0                                                                                                
            return (h + m / 60.0) % 24.0                                                                             
        except Exception:                                                                                            
            return np.nan                                                                                            
                                                                                                                    
    out.loc[miss] = cleaned.map(_parse_one)                                                                          
    return out                                                                                                       
                                                                                                                    
sri_src = df_params[["SEQN", "sample_idx", "SLQ300", "SLQ310", "SLQ320", "SLQ330"]].copy()
sri_src["sample_idx"] = pd.to_numeric(sri_src["sample_idx"], errors="coerce")                                        
sri_src = sri_src.dropna(subset=["SEQN", "sample_idx"]).copy()                                                       
sri_src["sample_idx"] = sri_src["sample_idx"].astype(int)                                                            
                                                                                                                    
for c in ["SLQ300", "SLQ310", "SLQ320", "SLQ330"]:                                                                   
    sri_src[c + "_h"] = parse_time_series_to_hour(sri_src[c])                                                        
                                                                                                                    
dow = ((sri_src["sample_idx"].to_numpy() - 1) % 7) + 1                                                               
weekend_mask = np.isin(dow, list(WEEKEND_DOW))                                                                       
                                                                                                                    
bed = np.where(weekend_mask, sri_src["SLQ320_h"].to_numpy(), sri_src["SLQ300_h"].to_numpy())                         
wake = np.where(weekend_mask, sri_src["SLQ330_h"].to_numpy(), sri_src["SLQ310_h"].to_numpy())                        
sri_src["bed_h"] = bed                                                                                               
sri_src["wake_h"] = wake                                                                                             
sri_src = sri_src[np.isfinite(sri_src["bed_h"]) & np.isfinite(sri_src["wake_h"])].copy()                             
if sri_src.empty:                                                                                                    
    raise ValueError("SRI 계산용 유효 bed/wake 데이터가 없습니다.")                                                  
                                                                                                                    
sri_src = sri_src.sort_values(["SEQN", "sample_idx"], kind="mergesort").reset_index(drop=True)                       
                                                                                                                    
bins_per_day = int(24 * 60 // EPOCH_MIN)                                                                             
t = np.arange(bins_per_day, dtype=float) * (EPOCH_MIN / 60.0)                                                        
                                                                                                                    
bed_arr = sri_src["bed_h"].to_numpy(dtype=float)[:, None]                                                            
wake_arr = sri_src["wake_h"].to_numpy(dtype=float)[:, None]                                                          
cross_midnight = wake_arr < bed_arr                                                                                  
                                                                                                                    
asleep = np.where(                                                                                                   
    cross_midnight,                                                                                                  
    (t >= bed_arr) | (t < wake_arr),                                                                                 
    (t >= bed_arr) & (t < wake_arr),
).astype(np.uint8)                                                                                                   
                                                                                                                    
seqn = sri_src["SEQN"].to_numpy()                                                                                    
sidx = sri_src["sample_idx"].to_numpy()                                                                              
pair_mask = (seqn[1:] == seqn[:-1]) & ((sidx[1:] - sidx[:-1]) == 1)                                                  
if not pair_mask.any():                                                                                              
    raise ValueError("SRI 계산용 연속일(pair) 데이터가 없습니다. sample_idx를 확인하세요.")                          
                                                                                                                    
pair_same_ratio = (asleep[1:] == asleep[:-1]).mean(axis=1) * 100.0                                                   
pairs_df = pd.DataFrame({"SEQN": seqn[1:][pair_mask], "pair_sri": pair_same_ratio[pair_mask]})                       
sri_df = pairs_df.groupby("SEQN", as_index=False)["pair_sri"].mean().rename(columns={"pair_sri": "SRI"})             
                                                                                                                    
if sri_df.empty:                                                                                                     
    raise ValueError("개인별 SRI 계산 결과가 비어 있습니다.")                                                        
                                                                                                                    
sleep_pattern_score_baseline = float(np.nanpercentile(sri_df["SRI"], SRI_BASELINE_PERCENTILE))                       
sleep_pattern_score_post = float(np.nanpercentile(sri_df["SRI"], SRI_POST_PERCENTILE))                               
                                                                                                                    
theta_sleep_pattern, delta_e_sleep_pattern = calc_theta_from_rr(                                                     
    sleep_pattern_score_baseline, sleep_pattern_score_post, rr_sleep_pattern, DELTA_E_SCALE                          
)                                                                                                                    
                                                                                                                    
# -------------------------                                                                                          
# 6) 최종 theta                                                                                                      
# -------------------------                                                                                          
theta = float(np.mean([theta_step, theta_sleep, theta_sleep_pattern]))                                               
                                                                                                                    
# -------------------------                                                                                          
# 7) 출력                                                                                                            
# -------------------------                                                                                          
print(f"USE_NOTEBOOK_SCALE={USE_NOTEBOOK_SCALE} (DELTA_E_SCALE={DELTA_E_SCALE})")                                    
print("data source for weighting/SRI: df_params\n")                                                                  
                                                                                                                    
print("[STEP]")                                                                                                      
print(f"  raw: {step_raw_baseline:.1f} -> {step_raw_post:.1f}")                                                      
print(f"  score: {step_score_baseline:.6f} -> {step_score_post:.6f} (delta_E={delta_e_step:.6f})")                   
print(f"  theta_step = {theta_step:.6f}\n")                                                                          

print("[SLEEP TIME]")
print(f"  hours: {sleep_hours_baseline:.2f} -> {sleep_hours_post:.2f}")                                              
print(f"  score: {sleep_score_baseline:.6f} -> {sleep_score_post:.6f} (delta_E={delta_e_sleep:.6f})")                
print(f"  theta_sleep = {theta_sleep:.6f}\n")                                                                        
                                                                                                                    
print("[SLEEP PATTERN | SRI percentile]")                                                                            
print(f"  SRI p{SRI_BASELINE_PERCENTILE:.0f} -> p{SRI_POST_PERCENTILE:.0f}: "                                        
    f"{sleep_pattern_score_baseline:.6f} -> {sleep_pattern_score_post:.6f}")                                       
print(f"  delta_E={delta_e_sleep_pattern:.6f}")                                                                      
print(f"  theta_sleep_pattern = {theta_sleep_pattern:.6f}\n")                                                        
                                                                                                                    
print(f"theta(평균) = {theta:.6f}")                                                                                  
                                                                                                                    
# 필요 시 상세 확인
# display(step_detail.sort_values('age_group').reset_index(drop=True))                                               
# display(sleep_detail.sort_values('age_group').reset_index(drop=True))                                              
# display(sri_df.head(20))

USE_NOTEBOOK_SCALE=True (DELTA_E_SCALE=300.0)
data source for weighting/SRI: df_params

[STEP]
  raw: 2000.0 -> 7000.0
  score: 12.041452 -> 89.982390 (delta_E=0.259803)
  theta_step = 1.107308

[SLEEP TIME]
  hours: 5.00 -> 7.00
  score: 10.000000 -> 100.000000 (delta_E=0.300000)
  theta_sleep = 0.690047

[SLEEP PATTERN | SRI percentile]
  SRI p25 -> p75: 52.873563 -> 59.267241
  delta_E=0.021312
  theta_sleep_pattern = 10.844074

theta(평균) = 4.213810


In [15]:
import numpy as np
import pandas as pd

# sri_df가 이미 생성되어 있다고 가정 (컬럼: "SRI")
if "sri_df" not in globals() or sri_df.empty:
    raise ValueError("sri_df가 없거나 비어 있습니다. SRI 계산 셀을 먼저 실행하세요.")

# 1) 현재 시나리오(기본: p25 -> p75) sleep_pattern 값
baseline_pct = float(globals().get("SRI_BASELINE_PERCENTILE", 25.0))
post_pct = float(globals().get("SRI_POST_PERCENTILE", 75.0))

sleep_pattern_baseline = float(np.nanpercentile(sri_df["SRI"], baseline_pct))
sleep_pattern_post = float(np.nanpercentile(sri_df["SRI"], post_pct))

print("[SLEEP PATTERN | 시나리오 값]")
print(f"  p{baseline_pct:.0f}: {sleep_pattern_baseline:.6f}")
print(f"  p{post_pct:.0f}: {sleep_pattern_post:.6f}")
print(f"  변화량: {sleep_pattern_post - sleep_pattern_baseline:.6f}\n")

# 2) SRI 분포 전체에서의 sleep_pattern 값 테이블
percentiles = [0, 5, 10, 25, 50, 75, 90, 95, 100]
values = np.nanpercentile(sri_df["SRI"], percentiles)

sleep_pattern_by_sri_dist = pd.DataFrame({
    "percentile": percentiles,
    "sleep_pattern_value": values
})

print("[SRI 분포별 sleep_pattern 값]")
display(sleep_pattern_by_sri_dist)

[SLEEP PATTERN | 시나리오 값]
  p25: 52.873563
  p75: 59.267241
  변화량: 6.393678

[SRI 분포별 sleep_pattern 값]


,percentile,sleep_pattern_value
0,0,38.793103
1,5,48.275862
2,10,50.000000
3,25,52.873563
4,50,56.106322
5,75,59.267241
6,90,61.997126
7,95,63.717672
8,100,72.413793


In [16]:
# import numpy as np

# # =========================
# # theta 계산 입력값 (직접 하드코딩)
# # =========================

# # True: 현재 노트북 로직(총점 0~300, delta_E=(변화량)/300) 기준
# # False: 질문의 예시 정의(도메인 점수 0~100, delta_E=(변화량)/100) 기준
# USE_NOTEBOOK_SCALE = True
# DELTA_E_SCALE = 300.0 if USE_NOTEBOOK_SCALE else 100.0

# # [Step] 2,000보 -> 7,000보 개선 시
# step_before, step_after = 50.0, 70.0
# rr_step = 0.75  # 문헌: The Lancet (2022)

# # [Sleep Time] 5시간 미만 -> 7~8시간 확보 시
# sleep_before, sleep_after = 50.0, 70.0
# rr_sleep = 1/1.23   # 문헌: Tao et al.(2021, UK Biobank)

# # [Sleep Pattern] 불규칙 패턴 -> 규칙적 패턴 교정 시
# sleep_pattern_before, sleep_pattern_after = 50.0, 70.0
# rr_sleep_pattern = 1/1.26  # 문헌: Chaput et al.(2024, JECH)


# def calc_theta_from_rr(before_score, after_score, rr, scale):
#     delta_e = (float(after_score) - float(before_score)) / float(scale)
#     if rr <= 0 or rr >= 1:
#         raise ValueError(f"RR은 (0, 1) 범위여야 합니다. 입력값: {rr}")
#     if delta_e <= 0:
#         raise ValueError(f"delta_E는 0보다 커야 합니다. 입력값: {delta_e}")
#     return -np.log(rr) / delta_e


# theta_step = calc_theta_from_rr(step_before, step_after, rr_step, DELTA_E_SCALE)
# theta_sleep = calc_theta_from_rr(sleep_before, sleep_after, rr_sleep, DELTA_E_SCALE)
# theta_sleep_pattern = calc_theta_from_rr(
#     sleep_pattern_before, sleep_pattern_after, rr_sleep_pattern, DELTA_E_SCALE
# )

# print(f"USE_NOTEBOOK_SCALE={USE_NOTEBOOK_SCALE} (DELTA_E_SCALE={DELTA_E_SCALE})")
# print(f"theta_step = {theta_step:.6f}")
# print(f"theta_sleep = {theta_sleep:.6f}")
# print(f"theta_sleep_pattern = {theta_sleep_pattern:.6f}")
# print(f"theta(평균, 참고) = {np.mean([theta_step, theta_sleep, theta_sleep_pattern]):.6f}")

In [17]:
import pandas as pd
import numpy as np

# 1) 기존 데이터 가져오기 (마지막 시뮬레이션 결과 기준)
if 'pricing_df' in globals():
    df_base = pricing_df.copy()
elif 'seqn_premium' in globals():
    df_base = seqn_premium.copy()
else:
    # 데이터가 없으면 master_report_context에서 추출 시도
    if 'master_report_context' in globals():
        df_base = master_report_context.get('pricing_for_report', pd.DataFrame())
    
if df_base.empty:
    print("오류: 시뮬레이션 결과 데이터(pricing_df 등)를 찾을 수 없습니다. 시뮬레이션 셀을 먼저 실행해 주세요.")
else:
    # 2) 시나리오 설정 (L_used 변경)
    L_scenarios = [0.05, 0.10, 0.15, 0.20]
    results = []

    # 필요한 파라미터 가져오기
    k_val = globals().get('k_used', globals().get('k_opt', 0.03))
    E0_val = globals().get('E0_used', globals().get('E0_opt', 150.0))
    severity_val = globals().get('severity', 10000000.0)
    op_cost = globals().get('operating_cost', 0.0)

    for L in L_scenarios:
        temp = df_base.copy()
        
        # 할인율 및 할인액 재계산
        # d(E) = L / (1 + exp(-k(E - E0)))
        effort = temp['effort_score_E'] if 'effort_score_E' in temp.columns else temp['monthly_avg_total_score']
        d_rate = L / (1.0 + np.exp(-k_val * (effort - E0_val)))
        
        # 할인액 (상한 적용 포함)
        raw_discount = temp['base_premium_P0'] * d_rate
        max_dist = globals().get('max_discount_amount', 0)
        if max_dist > 0:
            applied_discount = raw_discount.clip(upper=max_dist)
        else:
            applied_discount = raw_discount
            
        premium = temp['base_premium_P0'] - applied_discount
        
        # 기존 기대손해액 (보험금 지급액)
        exp_loss_claims = temp['expected_loss_E_L'] if 'expected_loss_E_L' in temp.columns else (temp['lambda_effective'] * severity_val)
        
        # <<< 사용자가 생각한 "확장된 기대손해액" >>>
        # 보험사 입장에서 나가는 총 비용 = 지급 보험금 + 고객에게 깎아준 할인액
        total_expenditure = exp_loss_claims + applied_discount
        
        profit = premium - exp_loss_claims - op_cost
        
        results.append({
            '할인 상한(L_used)': L,
            '평균 보험료(원)': premium.mean(),
            '평균 기대손해액(순수 보험금)': exp_loss_claims.mean(),
            '평균 할인액(비용)': applied_discount.mean(),
            '사용자 정의 기대손해액(보험금+할인액)': total_expenditure.mean(),
            '평균 기대이익(원)': profit.mean()
        })

    # 3) 결과 테이블 출력
    summary_df = pd.DataFrame(results)
    
    # pandas style (jinja2) 라이브러리 없이 수동 문자열 포맷팅
    display_df = summary_df.copy()
    display_df['할인 상한(L_used)'] = display_df['할인 상한(L_used)'].apply(lambda x: f"{x:.0%}")
    for col in display_df.columns:
        if col != '할인 상한(L_used)':
            display_df[col] = display_df[col].apply(lambda x: f"{x:,.0f}")

    print("📊 [사용자 정의] 할인 상한 변화에 따른 보험사 지출 분석")
    print("- '사용자 정의 기대손해액'은 순수 보험금에 '할인해준 금액'을 비용으로 간주하여 합산한 값입니다.")
    display(display_df)

📊 [사용자 정의] 할인 상한 변화에 따른 보험사 지출 분석
- '사용자 정의 기대손해액'은 순수 보험금에 '할인해준 금액'을 비용으로 간주하여 합산한 값입니다.


,할인 상한(L_used),평균 보험료(원),평균 기대손해액(순수 보험금),평균 할인액(비용),사용자 정의 기대손해액(보험금+할인액),평균 기대이익(원)
0,5%,"60,862","6,670","1,355","8,025","54,192"
1,10%,"59,508","6,670","2,709","9,379","52,838"
2,15%,"58,153","6,670","4,064","10,734","51,483"
3,20%,"56,798","6,670","5,418","12,088","50,129"


In [18]:
import numpy as np
import pandas as pd                                                                                                                                                                                                                                                               
                                                                                                                                                                                                                                                                                
# =========================================================                                                                                                                                                                                                                       
# 인과형(증분 노력) ROI: 할인 상한 5/10/15/20% 비교                                                                                                                                                                                                                               
# - 노트북 현재 값 사용                                                                                                                                                                                                                                                           
# - eta는 ETA_INCENTIVE 있으면 우선, 없으면 EFFORT_BETA, 그것도 없으면 0.2                                                                                                                                                                                                        
# =========================================================
                                                                                                                                                                                                                                                                                
# 0) helper/입력 체크                                                                                                                                                                                                                                                             
need_funcs = ["_get_pricing_df", "_first_available", "_safe_float"]                                                                                                                                                                                                               
missing = [f for f in need_funcs if f not in globals()]                                                                                                                                                                                                                           
if missing:                                                                                                                                                                                                                                                                       
    raise ValueError(f"필수 helper 함수 누락: {missing}")                                                                                                                                                                                                                         
                                                                                                                                                                                                                                                                                
df0 = _get_pricing_df().copy()                                                                                                                                                                                                                                                    
if not isinstance(df0, pd.DataFrame) or df0.empty:                                                                                                                                                                                                                                
    raise ValueError("pricing_for_report(_get_pricing_df 결과)가 비어 있습니다.")                                                                                                                                                                                                 
                                                                                                                                                                                                                                                                                
# effort 컬럼 정합                                                                                                                                                                                                                                                                
if "effort_score_E" not in df0.columns and "monthly_avg_total_score" in df0.columns:                                                                                                                                                                                              
    df0["effort_score_E"] = pd.to_numeric(df0["monthly_avg_total_score"], errors="coerce")                                                                                                                                                                                        
if "effort_score_E" not in df0.columns:                                                                                                                                                                                                                                           
    raise ValueError("effort_score_E(또는 monthly_avg_total_score)가 필요합니다.")                                                                                                                                                                                                
                                                                                                                                                                                                                                                                                
# 필수 컬럼                                                                                                                                                                                                                                                                       
for c in ["base_premium_P0", "lambda0"]:                                                                                                                                                                                                                                          
    if c not in df0.columns:                                                                                                                                                                                                                                                      
        raise ValueError(f"`{c}` 컬럼이 필요합니다.")                                                                                                                                                                                                                             
                                                                                                                                                                                                                                                                                
# 1) 노트북 현재 파라미터 로드                                                                                                                                                                                                                                                    
k_used = _safe_float(_first_available(["k_used", "k_opt", "discount_k"], np.nan))                                                                                                                                                                                                 
E0_used = _safe_float(_first_available(["E0_used", "E0_opt", "discount_E0"], np.nan))                                                                                                                                                                                             
theta_used = _safe_float(_first_available(["theta"], np.mean([                                                                                                                                                                                                                    
    _safe_float(_first_available(["theta_step"], 5.0), 5.0),                                                                                                                                                                                                                      
    _safe_float(_first_available(["theta_sleep"], 5.0), 5.0),                                                                                                                                                                                                                     
    _safe_float(_first_available(["theta_sleep_pattern"], 5.0), 5.0),                                                                                                                                                                                                             
])), 5.0)                                                                                                                                                                                                                                                                         
severity_used = _safe_float(_first_available(["severity"], 10_000_000.0), 10_000_000.0)                                                                                                                                                                                           
max_discount_amount = _safe_float(_first_available(["max_discount_amount"], 0.0), 0.0)                                                                                                                                                                                            
                                                                                                                                                                                                                                                                                
# 행동탄력도(노트북 값 우선 사용)                                                                                                                                                                                                                                                 
# eta = float(globals().get("ETA_INCENTIVE", globals().get("EFFORT_BETA", 1)))    
eta = 1                                                                                                                                                                                             
                                                                                                                                                                                                                                                                                
for name, v in {                                                                                                                                                                                                                                                                  
    "k_used": k_used, "E0_used": E0_used, "theta_used": theta_used, "severity_used": severity_used                                                                                                                                                                                
}.items():                                                                                                                                                                                                                                                                        
    if not np.isfinite(v):                                                                                                                                                                                                                                                        
        raise ValueError(f"{name} 값이 유효하지 않습니다: {v}")                                                                                                                                                                                                                   
                                                                                                                                                                                                                                                                                
# 2) 기본 데이터 정리                                                                                                                                                                                                                                                             
base = df0.copy()                                                                                                                                                                                                                                                                 
base["effort_score_E"] = pd.to_numeric(base["effort_score_E"], errors="coerce").fillna(0.0).clip(0, 300)
base["base_premium_P0"] = pd.to_numeric(base["base_premium_P0"], errors="coerce").fillna(0.0)                                                                                                                                                                                     
base["lambda0"] = pd.to_numeric(base["lambda0"], errors="coerce").fillna(0.0)                                                                                                                                                                                                     
                                                                                                                                                                                                                                                                                
# 기준 시나리오 L=0                                                                                                                                                                                                                                                               
base["discount_rate_0"] = 0.0                                                                                                                                                                                                                                                     
base["discount_amount_0"] = 0.0                                                                                                                                                                                                                                                   

# 3) 시나리오 계산                                                                                                                                                                                                                                                                
L_scenarios = [0.05, 0.10, 0.15, 0.20]                                                                                                                                                                                                                                            
rows = []                                                                                                                                                                                                                                                                         
causal_rows = {}                                                                                                                                                                                                                                                                  
                                                                                                                                                                                                                                                                                
for L in L_scenarios:                                                                                                                                                                                                                                                             
    df = base.copy()                                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                                                
    # 할인율 (현재 노력점수 기준)                                                                                                                                                                                                                                                 
    df["discount_rate_L"] = np.clip(                                                                                                                                                                                                                                              
        L / (1.0 + np.exp(-k_used * (df["effort_score_E"] - E0_used))),                                                                                                                                                                                                           
        0.0, 1.0                                                                                                                                                                                                                                                                  
    )                                                                                                                                                                                                                                                                             
    df["raw_discount_amount_L"] = df["base_premium_P0"] * df["discount_rate_L"]                                                                                                                                                                                                   
                                                                                                                                                                                                                                                                                
    if max_discount_amount > 0:                                                                                                                                                                                                                                                   
        df["discount_amount_L"] = df["raw_discount_amount_L"].clip(upper=max_discount_amount)                                                                                                                                                                                     
    else:                                                                                                                                                                                                                                                                         
        df["discount_amount_L"] = df["raw_discount_amount_L"]                                                                                                                                                                                                                     
                                                                                                                                                                                                                                                                                
    # 인센티브로 인한 증분 노력(가정)                                                                                                                                                                                                                                             
    # discount_rate는 비율(0~1)이므로 %p로 변환해 eta 적용                                                                                                                                                                                                                        
    df["effort_increment"] = eta * (df["discount_rate_L"] - df["discount_rate_0"]) * 100.0                                                                                                                                                                                        
    df["effort_cf"] = (df["effort_score_E"] + df["effort_increment"]).clip(0, 300)                                                                                                                                                                                                
                                                                                                                                                                                                                                                                                
    # 증분 노력만으로 발생한 추가 리스크 감소                                                                                                                                                                                                                                     
    df["deltaE_increment"] = np.maximum(0.0, df["effort_cf"] - df["effort_score_E"]) / 300.0                                                                                                                                                                                      
    df["benefit_increment"] = (                                                                                                                                                                                                                                                   
        df["lambda0"] * (1.0 - np.exp(-theta_used * df["deltaE_increment"])) * severity_used                                                                                                                                                                                      
    )                                                                                                                                                                                                                                                                             
                                                                                                                                                                                                                                                                                
    # 비용/순효과/ROI                                                                                                                                                                                                                                                             
    df["cost_increment"] = df["discount_amount_L"] - df["discount_amount_0"]  # = discount_amount_L                                                                                                                                                                               
    df["net_increment"] = df["benefit_increment"] - df["cost_increment"]
    df["roi_increment"] = np.where(df["cost_increment"] > 0, df["benefit_increment"] / df["cost_increment"], np.nan)                                                                                                                                                              
                                                                                                                                                                                                                                                                                
    avg_benefit = float(pd.to_numeric(df["benefit_increment"], errors="coerce").mean())                                                                                                                                                                                           
    avg_cost = float(pd.to_numeric(df["cost_increment"], errors="coerce").mean())                                                                                                                                                                                                 
    avg_net = float(pd.to_numeric(df["net_increment"], errors="coerce").mean())                                                                                                                                                                                                   
    roi_mean = float(pd.to_numeric(df["roi_increment"], errors="coerce").mean())                                                                                                                                                                                                  
    total_net = float(pd.to_numeric(df["net_increment"], errors="coerce").sum())                                                                                                                                                                                                  
    share_pos = float((pd.to_numeric(df["net_increment"], errors="coerce") > 0).mean())                                                                                                                                                                                           
                                                                                                                                                                                                                                                                                
    rows.append({                                                                                                                                                                                                                                                                 
        "discount_cap_pct": L * 100.0,                                                                                                                                                                                                                                            
        "eta_used": eta,                                                                                                                                                                                                                                                          
        "avg_benefit_increment": avg_benefit,                                                                                                                                                                                                                                     
        "avg_discount_cost": avg_cost,                                                                                                                                                                                                                                            
        "avg_net_increment": avg_net,                                                                                                                                                                                                                                             
        "roi_increment_mean": roi_mean,                                                                                                                                                                                                                                           
        "total_net_increment": total_net,                                                                                                                                                                                                                                         
        "share_net_positive": share_pos,                                                                                                                                                                                                                                          
        "is_profitable_total_net_gt_0": bool(total_net > 0),                                                                                                                                                                                                                      
        "is_roi_gt_1": bool(np.isfinite(roi_mean) and roi_mean > 1),                                                                                                                                                                                                              
    })                                                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                                                
    keep_cols = [c for c in [                                                                                                                                                                                                                                                     
        "bootstrap_id", "SEQN", "Cluster",                                                                                                                                                                                                                                        
        "effort_score_E", "effort_increment", "effort_cf", "deltaE_increment",                                                                                                                                                                                                    
        "discount_rate_L", "discount_amount_L",                                                                                                                                                                                                                                   
        "benefit_increment", "cost_increment", "net_increment", "roi_increment"                                                                                                                                                                                                   
    ] if c in df.columns]                                                                                                                                                                                                                                                         
    causal_rows[L] = df[keep_cols].copy()                                                                                                                                                                                                                                         
                                                                                                                                                                                                                                                                                
causal_roi_df = pd.DataFrame(rows).sort_values("discount_cap_pct").reset_index(drop=True)                                                                                                                                                                                         
                                                                                                                                                                                                                                                                                
print("=== 인과형(증분 노력) ROI 비교 ===")                                                                                                                                                                                                                                       
print(f"eta(노트북값) = {eta}")                                                                                                                                                                                                                                                   
for _, r in causal_roi_df.iterrows():                                                                                                                                                                                                                                             
    print(                                                                                                                                                                                                                                                                        
        f"[상한 {r['discount_cap_pct']:.0f}%] "                                                                                                                                                                                                                                   
        f"증분편익={r['avg_benefit_increment']:,.0f}원, "                                                                                                                                                                                                                         
        f"증분비용={r['avg_discount_cost']:,.0f}원, "                                                                                                                                                                                                                             
        f"순효과={r['avg_net_increment']:,.0f}원, "                                                                                                                                                                                                                               
        f"ROI={r['roi_increment_mean']:.3f}, "                                                                                                                                                                                                                                    
        f"총순효과={r['total_net_increment']:,.0f}원"                                                                                                                                                                                                                             
    )                                                                                                                                                                                                                                                                             

display(causal_roi_df)                                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                                                
# 필요 시 상세 확인:                                                                                                                                                                                                                                                              
# display(causal_rows[0.05].head(20))                                                                                                                                                                                                                                             
# display(causal_rows[0.10].head(20))                                                                                                                                                                                                                                             
# display(causal_rows[0.15].head(20))                                                                                                                                                                                                                                             
# display(causal_rows[0.20].head(20)) 

=== 인과형(증분 노력) ROI 비교 ===
eta(노트북값) = 1
[상한 5%] 증분편익=258원, 증분비용=1,355원, 순효과=-1,096원, ROI=0.141, 총순효과=-109,614원
[상한 10%] 증분편익=506원, 증분비용=2,709원, 순효과=-2,204원, ROI=0.139, 총순효과=-220,370원
[상한 15%] 증분편익=742원, 증분비용=4,064원, 순효과=-3,322원, ROI=0.137, 총순효과=-332,207원
[상한 20%] 증분편익=968원, 증분비용=5,418원, 순효과=-4,451원, ROI=0.135, 총순효과=-445,065원


,discount_cap_pct,eta_used,avg_benefit_increment,avg_discount_cost,avg_net_increment,roi_increment_mean,total_net_increment,share_net_positive,is_profitable_total_net_gt_0,is_roi_gt_1
0,5.0,1,258.471595,1354.609619,-1096.138024,0.141302,-109613.802434,0.0,False,False
1,10.0,1,505.518479,2709.219238,-2203.700759,0.139204,-220370.075874,0.0,False,False
2,15.0,1,741.762921,4063.828857,-3322.065935,0.137161,-332206.593527,0.0,False,False
3,20.0,1,967.789804,5418.438475,-4450.648671,0.135173,-445064.867147,0.0,False,False


In [19]:
import numpy as np
import pandas as pd                                                                                                                                                                                                               
                                                                                                                                                                                                                                
# =========================================================                                                                                                                                                                       
# 내생적 노력-발생률 연동 시뮬레이션 (노트북 정합형)                                                                                                                                                                              
# - 할인 상한: 5%, 10%, 15%, 20%                                                                                                                                                                                                  
# - 노력(E)이 할인율에 반응하고, 그 E가 다시 발생률(lambda)에 반영됨                                                                                                                                                              
# =========================================================                                                                                                                                                                       
                                                                                                                                                                                                                                
# 0) 필수 helper 확인                                                                                                                                                                                                             
need_funcs = ["_get_pricing_df", "_first_available", "_safe_float"]                                                                                                                                                               
missing_funcs = [f for f in need_funcs if f not in globals()]
if missing_funcs:                                                                                                                                                                                                                 
    raise ValueError(f"필수 helper 함수 누락: {missing_funcs}")                                                                                                                                                                   
                                                                                                                                                                                                                                
# 1) 데이터 로드 (리포트와 동일 기준)                                                                                                                                                                                             
df = _get_pricing_df().copy()                                                                                                                                                                                                     
if not isinstance(df, pd.DataFrame) or df.empty:                                                                                                                                                                                  
    raise ValueError("pricing_for_report(_get_pricing_df 결과)가 비어 있습니다.")                                                                                                                                                 
                                                                                                                                                                                                                                
if "effort_score_E" not in df.columns and "monthly_avg_total_score" in df.columns:                                                                                                                                                
    df["effort_score_E"] = pd.to_numeric(df["monthly_avg_total_score"], errors="coerce")                                                                                                                                          
if "effort_score_E" not in df.columns:                                                                                                                                                                                            
    raise ValueError("effort_score_E(또는 monthly_avg_total_score)가 필요합니다.")                                                                                                                                                
                                                                                                                                                                                                                                
for c in ["base_premium_P0", "lambda0"]:                                                                                                                                                                                          
    if c not in df.columns:                                                                                                                                                                                                       
        raise ValueError(f"`{c}` 컬럼이 필요합니다.")                                                                                                                                                                             
                                                                                                                                                                                                                                
df["effort_score_E"] = pd.to_numeric(df["effort_score_E"], errors="coerce").fillna(0.0).clip(0, 300)                                                                                                                              
df["base_premium_P0"] = pd.to_numeric(df["base_premium_P0"], errors="coerce").fillna(0.0)                                                                                                                                         
df["lambda0"] = pd.to_numeric(df["lambda0"], errors="coerce").fillna(0.0)                                                                                                                                                         
                                                                                                                                                                                                                                
# 2) 파라미터 (현재 노트북 값 사용)                                                                                                                                                                                               
k_used = _safe_float(_first_available(["k_used", "k_opt", "discount_k"], np.nan))                                                                                                                                                 
E0_used = _safe_float(_first_available(["E0_used", "E0_opt", "discount_E0"], np.nan))                                                                                                                                             
theta_used = _safe_float(_first_available(["theta"], np.mean([                                                                                                                                                                    
    _safe_float(_first_available(["theta_step"], 5.0), 5.0),
    _safe_float(_first_available(["theta_sleep"], 5.0), 5.0),                                                                                                                                                                     
    _safe_float(_first_available(["theta_sleep_pattern"], 5.0), 5.0),                                                                                                                                                             
])), 5.0)                                                                                                                                                                                                                         
severity_used = _safe_float(_first_available(["severity"], 10_000_000.0), 10_000_000.0)                                                                                                                                           
operating_cost_used = _safe_float(_first_available(["operating_cost"], 0.0), 0.0)
max_discount_amount = _safe_float(_first_available(["max_discount_amount"], 0.0), 0.0)                                                                                                                                            
                                                                                                                                                                                                                                
# 할인 -> 노력 반응 계수(노트북 값 우선)                                                                                                                                                                                          
eta_used = 1                                                                                                                                         
                                                                                                                                                                                                                                
for n, v in {                                                                                                                                                                                                                     
    "k_used": k_used, "E0_used": E0_used, "theta_used": theta_used, "severity_used": severity_used                                                                                                                                
}.items():                                                                                                                                                                                                                        
    if not np.isfinite(v):                                                                                                                                                                                                        
        raise ValueError(f"{n} 값이 유효하지 않습니다: {v}")                                                                                                                                                                      
                                                                                                                                                                                                                                
# 3) 베이스라인 (L=0, 무할인/무유도)                                                                                                                                                                                              
base = df.copy()                                                                                                                                                                                                                  
base["E_base"] = base["effort_score_E"]                                                                                                                                                                                           
base["delta_E_base"] = np.maximum(0.0, base["E_base"] - E0_used) / 300.0                                                                                                                                                          
base["lambda_eff_base"] = base["lambda0"] * np.exp(-theta_used * base["delta_E_base"])                                                                                                                                            
base["expected_loss_base"] = base["lambda_eff_base"] * severity_used                                                                                                                                                              
base["discount_amount_base"] = 0.0                                                                                                                                                                                                
base["premium_base"] = base["base_premium_P0"]                                                                                                                                                                                    
base["profit_base"] = base["premium_base"] - base["expected_loss_base"] - operating_cost_used                                                                                                                                     
                                                                                                                                                                                                                                
total_profit_base = float(pd.to_numeric(base["profit_base"], errors="coerce").sum())                                                                                                                                              
                                                                                                                                                                                                                                
# 4) 시나리오 계산                                                                                                                                                                                                                
L_scenarios = [0.05, 0.10, 0.15, 0.20]                                                                                                                                                                                            
endogenous_rows = {}                                                                                                                                                                                                              
summary_rows = []                                                                                                                                                                                                                 
                                                                                                                                                                                                                                
for L in L_scenarios:                                                                                                                                                                                                             
    s = base.copy()                                                                                                                                                                                                               
                                                                                                                                                                                                                                
    # 현재 노력 기준 할인율                                                                                                                                                                                                       
    s["discount_rate"] = np.clip(                                                                                                                                                                                                 
        L / (1.0 + np.exp(-k_used * (s["E_base"] - E0_used))),                                                                                                                                                                    
        0.0, 1.0                                                                                                                                                                                                                  
    )                                                                                                                                                                                                                             
    s["raw_discount_amount"] = s["base_premium_P0"] * s["discount_rate"]                                                                                                                                                          
    if max_discount_amount > 0:
        s["discount_amount"] = s["raw_discount_amount"].clip(upper=max_discount_amount)                                                                                                                                           
    else:                                                                                                                                                                                                                         
        s["discount_amount"] = s["raw_discount_amount"]                                                                                                                                                                           
                                                                                                                                                                                                                                
    # 내생적 노력 업데이트: E_new = E_base + eta * (할인율 %p)                                                                                                                                                                    
    s["E_new"] = (s["E_base"] + eta_used * (s["discount_rate"] * 100.0)).clip(0, 300)                                                                                                                                             
                                                                                                                                                                                                                                
    # 업데이트된 노력으로 발생률 재계산                                                                                                                                                                                           
    s["delta_E_endog"] = np.maximum(0.0, s["E_new"] - E0_used) / 300.0                                                                                                                                                            
    s["lambda_eff_endog"] = s["lambda0"] * np.exp(-theta_used * s["delta_E_endog"])                                                                                                                                               
    s["expected_loss_endog"] = s["lambda_eff_endog"] * severity_used                                                                                                                                                              
                                                                                                                                                                                                                                
    # 보험료/이익                                                                                                                                                                                                                 
    s["premium_endog"] = s["base_premium_P0"] - s["discount_amount"]                                                                                                                                                              
    s["profit_endog"] = s["premium_endog"] - s["expected_loss_endog"] - operating_cost_used                                                                                                                                       
                                                                                                                                                                                                                                
    # 보험사 관점 편익/비용/순효과                                                                                                                                                                                                
    s["benefit_risk_reduction"] = s["expected_loss_base"] - s["expected_loss_endog"]                                                                                                                                              
    s["cost_discount"] = s["discount_amount"]                                                                                                                                                                                     
    s["net_effect"] = s["benefit_risk_reduction"] - s["cost_discount"]                                                                                                                                                            
    s["roi"] = np.where(s["cost_discount"] > 0, s["benefit_risk_reduction"] / s["cost_discount"], np.nan)                                                                                                                         
                                                                                                                                                                                                                                
    avg_benefit = float(pd.to_numeric(s["benefit_risk_reduction"], errors="coerce").mean())                                                                                                                                       
    avg_cost = float(pd.to_numeric(s["cost_discount"], errors="coerce").mean())                                                                                                                                                   
    avg_net = float(pd.to_numeric(s["net_effect"], errors="coerce").mean())                                                                                                                                                       
    roi_mean = float(pd.to_numeric(s["roi"], errors="coerce").mean())                                                                                                                                                             
                                                                                                                                                                                                                                
    total_benefit = float(pd.to_numeric(s["benefit_risk_reduction"], errors="coerce").sum())                                                                                                                                      
    total_cost = float(pd.to_numeric(s["cost_discount"], errors="coerce").sum())                                                                                                                                                  
    total_net = float(pd.to_numeric(s["net_effect"], errors="coerce").sum())                                                                                                                                                      
                                                                                                                                                                                                                                
    total_profit_endog = float(pd.to_numeric(s["profit_endog"], errors="coerce").sum())                                                                                                                                           
    delta_total_profit_vs_base = total_profit_endog - total_profit_base                                                                                                                                                           
                                                                                                                                                                                                                                
    summary_rows.append({                                                                                                                                                                                                         
        "discount_cap_pct": L * 100.0,                                                                                                                                                                                            
        "eta_used": eta_used,                                                                                                                                                                                                     
        "avg_benefit_risk_reduction": avg_benefit,                                                                                                                                                                                
        "avg_discount_cost": avg_cost,                                                                                                                                                                                            
        "avg_net_effect": avg_net,                                                                                                                                                                                                
        "roi_mean": roi_mean,                                                                                                                                                                                                     
        "total_benefit_risk_reduction": total_benefit,                                                                                                                                                                            
        "total_discount_cost": total_cost,                                                                                                                                                                                        
        "total_net_effect": total_net,                                                                                                                                                                                            
        "total_profit_endog": total_profit_endog,                                                                                                                                                                                 
        "delta_total_profit_vs_base": delta_total_profit_vs_base,                                                                                                                                                                 
        "is_profitable_vs_base": bool(delta_total_profit_vs_base > 0),                                                                                                                                                            
    })                                                                                                                                                                                                                            
                                                                                                                                                                                                                                
    keep_cols = [c for c in [                                                                                                                                                                                                     
        "bootstrap_id", "SEQN", "Cluster",                                                                                                                                                                                        
        "E_base", "E_new", "delta_E_base", "delta_E_endog",                                                                                                                                                                       
        "discount_rate", "discount_amount",                                                                                                                                                                                       
        "lambda0", "lambda_eff_base", "lambda_eff_endog",                                                                                                                                                                         
        "expected_loss_base", "expected_loss_endog",                                                                                                                                                                              
        "benefit_risk_reduction", "cost_discount", "net_effect", "roi",                                                                                                                                                           
        "profit_base", "profit_endog"                                                                                                                                                                                             
    ] if c in s.columns]                                                                                                                                                                                                          
    endogenous_rows[L] = s[keep_cols].copy()                                                                                                                                                                                      
                                                                                                                                                                                                                                
endogenous_scenario_df = pd.DataFrame(summary_rows).sort_values("discount_cap_pct").reset_index(drop=True)
                                                                                                                                                                                                                                
print("=== 내생적 노력-발생률 연동 결과 ===")                                                                                                                                                                                     
print(f"eta_used = {eta_used}")                                                                                                                                                                                                   
print(f"baseline total_profit (L=0): {total_profit_base:,.0f} 원")                                                                                                                                                                
for _, r in endogenous_scenario_df.iterrows():                                                                                                                                                                                    
    print(                                                                                                                                                                                                                        
        f"[상한 {r['discount_cap_pct']:.0f}%] "                                                                                                                                                                                   
        f"평균편익={r['avg_benefit_risk_reduction']:,.0f}원, "                                                                                                                                                                    
        f"평균할인비용={r['avg_discount_cost']:,.0f}원, "
        f"평균순효과={r['avg_net_effect']:,.0f}원, "                                                                                                                                                                              
        f"ROI={r['roi_mean']:.3f}, "                                                                                                                                                                                              
        f"총이익증감(vs L=0)={r['delta_total_profit_vs_base']:,.0f}원"                                                                                                                                                            
    )                                                                                                                                                                                                                             
                                                                                                                                                                                                                                
display(endogenous_scenario_df)                                                                                                                                                                                                   
                                                                                                                                                                                                                                
# 필요 시 상세 행 확인                                                                                                                                                                                                            
# display(endogenous_rows[0.05].head(20))                                                                                                                                                                                         
# display(endogenous_rows[0.10].head(20))                                                                                                                                                                                         
# display(endogenous_rows[0.15].head(20))                                                                                                                                                                                         
# display(endogenous_rows[0.20].head(20)) 

=== 내생적 노력-발생률 연동 결과 ===
eta_used = 1
baseline total_profit (L=0): 5,554,700 원
[상한 5%] 평균편익=65원, 평균할인비용=1,355원, 평균순효과=-1,289원, ROI=0.014, 총이익증감(vs L=0)=-128,934원
[상한 10%] 평균편익=126원, 평균할인비용=2,709원, 평균순효과=-2,583원, ROI=0.013, 총이익증감(vs L=0)=-258,284원
[상한 15%] 평균편익=184원, 평균할인비용=4,064원, 평균순효과=-3,880원, ROI=0.013, 총이익증감(vs L=0)=-388,021원
[상한 20%] 평균편익=237원, 평균할인비용=5,418원, 평균순효과=-5,181원, ROI=0.012, 총이익증감(vs L=0)=-518,122원


,discount_cap_pct,eta_used,avg_benefit_risk_reduction,avg_discount_cost,avg_net_effect,roi_mean,total_benefit_risk_reduction,total_discount_cost,total_net_effect,total_profit_endog,delta_total_profit_vs_base,is_profitable_vs_base
0,5.0,1,65.265574,1354.609619,-1289.344045,0.013632,6526.557430,135460.961884,-128934.404454,5.425765e+06,-128934.404454,False
1,10.0,1,126.383745,2709.219238,-2582.835492,0.013199,12638.374550,270921.923767,-258283.549218,5.296416e+06,-258283.549218,False
2,15.0,1,183.618067,4063.828857,-3880.210790,0.012784,18361.806685,406382.885651,-388021.078966,5.166679e+06,-388021.078966,False
3,20.0,1,237.215344,5418.438475,-5181.223132,0.012387,23721.534368,541843.847535,-518122.313167,5.036578e+06,-518122.313167,False
